In [ ]:
import json
import os
import pickle
import subprocess
import sys
import sysconfig
import time
from datetime import datetime, timedelta
from pathlib import Path
from urllib.request import urlopen

# True only inside a real competition rerun; switches diagnostics + soft deadline.
TRUE_SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true"}
NOTEBOOK_START_EPOCH = time.time()

# Non-interactive matplotlib backend: diagnostics render plots with no display attached.
os.environ["MPLBACKEND"] = "Agg"
# Marks the run as a (real or emulated) submission so the framework + solver can adjust.
os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if TRUE_SUBMISSION else "0"
# Skip periodic JSON/HTML diagnostics and per-frame logging for every run.
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "1"

# Apply the measured vLLM winner before any serving setup command runs.
PUBLIC25_VLLM_PROFILE_NAME = 'kv5-bf16-mtp3-c8-cg32'
PUBLIC25_VLLM_PROFILE_ENV = {
    "TAAF_VLLM_ENABLE_PREFIX_CACHING": "0",
    "TAAF_VLLM_KV_CACHE_DTYPE": "auto",
    "TAAF_VLLM_KV_CACHE_MEMORY_BYTES": "5368709120",
    "TAAF_VLLM_MAX_CUDAGRAPH_CAPTURE_SIZE": "32",
    "TAAF_VLLM_MAX_NUM_BATCHED_TOKENS": "8192",
    "TAAF_VLLM_MAX_NUM_SEQS": "8",
    "TAAF_VLLM_MTP_TOKENS": "3",
    "TAAF_VLLM_OMP_THREADS": "1"
}
for key, value in PUBLIC25_VLLM_PROFILE_ENV.items():
    os.environ[key] = value
print(
    f'PUBLIC25_VLLM_PROFILE name={PUBLIC25_VLLM_PROFILE_NAME} '
    f'env={json.dumps(PUBLIC25_VLLM_PROFILE_ENV, sort_keys=True)}',
    flush=True,
)
# Pin arc_agi's cached level_reset_only before its client is built (RESET keeps the level).
os.environ["ONLY_RESET_LEVELS"] = "true"

# Prepend the CUDA toolkit to the linker path (it is off it on Kaggle GPU images) so the
# solver's GPU libraries (e.g. vllm / torch) can link against libcuda.
cuda_library_path = "/usr/local/nvidia/lib64"
os.environ["LIBRARY_PATH"] = os.pathsep.join(
    entry for entry in [cuda_library_path, *os.environ.get("LIBRARY_PATH", "").split(os.pathsep)] if entry
)

# Everything the run produces is written here.
WORKING_DIR = Path("/kaggle/working")
WORKING_DIR.mkdir(parents=True, exist_ok=True)
print(f"taaf.kaggle: TRUE_SUBMISSION={TRUE_SUBMISSION}")

In [ ]:
# Install the ARC runtime from the bundled competition wheels.
# Quiet: stdout is discarded; stderr (and a non-zero exit) still surface real failures.
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-index",
        "--no-warn-conflicts",
        "--disable-pip-version-check",
        "--find-links",
        "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels",
        "arc-agi",
    ],
    stdout=subprocess.DEVNULL,
)

In [ ]:
# Kaggle inputs attached to this notebook, plus bookkeeping paths used below.
DATASET_SOURCES = ["sergueimakarov/arc3-nextfork", "keithtyser/qwen38-flash-next-vllm-nvfp4-runtime-v1"]
KERNEL_SOURCES = []
DATASET_BUNDLE_MARKER = "taaf-kaggle-bundle.json"
SETUP_ENV_PATH = WORKING_DIR / "taaf_setup_env.json"


# Locate the source dataset by its marker file rather than a fixed mount path.
def _find_bundle_dir() -> Path:
    for marker in Path("/kaggle/input").rglob(DATASET_BUNDLE_MARKER):
        return marker.parent
    raise RuntimeError("TAAF source bundle not found under /kaggle/input.")


# Kaggle mounts a dataset at /kaggle/input/<slug> or /kaggle/input/datasets/<owner>/<slug>
# (depending on owner / slug collisions), so probe both and use whichever exists. Utility
# scripts mount under /kaggle/usr/lib/notebooks/<owner>/<slug>.
def _dataset_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [Path("/kaggle/input") / slug, Path("/kaggle/input/datasets") / owner / slug]


def _kernel_mount_candidates(ref: str) -> list[Path]:
    owner, slug = ref.split("/", 1)
    return [Path("/kaggle/usr/lib/notebooks") / owner / slug]


def _first_existing(candidates: list[Path]) -> Path | None:
    return next((c for c in candidates if c.exists()), None)


BUNDLE_DIR = _find_bundle_dir()
print(f"taaf.kaggle: source bundle = {BUNDLE_DIR}")

# Подключён ли ИМЕННО наш форк. Штамп кладёт scripts/build_nextfork_dataset.py;
# в чужом бандле его нет, поэтому подмена источника обнаруживается на первых
# секундах прогона, а не по бессмысленному результату через четыре часа.
_stamp = BUNDLE_DIR / "NEXTFORK_VERSION.txt"
if not _stamp.is_file():
    raise RuntimeError(
        f"NEXTFORK_VERSION.txt не найден в {BUNDLE_DIR}: подключён не наш форк. "
        f"Ожидался датасет {DATASET_SOURCES[0]}."
    )
print("nextfork: " + _stamp.read_text().strip().replace("\n", " | "))

# Map each attached input to where Kaggle actually mounted it (the source bundle is index 0).
kaggle_input_paths: dict[str, str] = {}
for i, ref in enumerate(DATASET_SOURCES):
    candidates = _dataset_mount_candidates(ref)
    resolved = BUNDLE_DIR if i == 0 else _first_existing(candidates)
    kaggle_input_paths[ref] = str(resolved or candidates[0])
for ref in KERNEL_SOURCES:
    candidates = _kernel_mount_candidates(ref)
    kaggle_input_paths[ref] = str(_first_existing(candidates) or candidates[0])

# Published to setup commands and the solver via the environment:
setup_env = {
    # JSON {ref: mount_path} so they can locate every attached dataset / utility script.
    "TAAF_KAGGLE_INPUT_PATHS": json.dumps(kaggle_input_paths, sort_keys=True),
    # The attached dataset refs in order (index 0 is this source bundle).
    "TAAF_KAGGLE_DATASET_SOURCES": json.dumps(DATASET_SOURCES),
    # The attached utility-script / kernel refs.
    "TAAF_KAGGLE_KERNEL_SOURCES": json.dumps(KERNEL_SOURCES),
}
os.environ.update(setup_env)
SETUP_ENV_PATH.write_text(json.dumps(setup_env, indent=2, sort_keys=True) + "\n")
print(f"taaf.kaggle: input paths = {setup_env['TAAF_KAGGLE_INPUT_PATHS']}")

In [ ]:
# Each bundled repo exposes its importable tree at <repo>/src or <repo>.
def _source_path_entries(bundle_dir: Path) -> list:
    entries = []
    for repo in sorted((bundle_dir / "src").iterdir(), reverse=True):
        for candidate in (repo / "src", repo):
            if candidate.is_dir():
                entries.append(candidate)
    return entries


# Environment handed to each setup command (paths + any keys it has persisted).
def _command_env() -> dict:
    env = os.environ.copy()
    # "$PYTHON" in a command resolves to this notebook's interpreter.
    env["PYTHON"] = sys.executable
    # Absolute path to the mounted source bundle.
    env["TAAF_KAGGLE_BUNDLE_DIR"] = str(BUNDLE_DIR)
    # The writable /kaggle/working directory.
    env["TAAF_KAGGLE_WORKING_DIR"] = str(WORKING_DIR)
    # A command writes a JSON object here to persist env keys to later commands + the run.
    env["TAAF_KAGGLE_SETUP_ENV"] = str(SETUP_ENV_PATH)
    env.update({str(k): str(v) for k, v in json.loads(SETUP_ENV_PATH.read_text()).items()})
    return env


# Make the bundled repos importable here (sys.path) and in child processes (.pth).
source_entries = _source_path_entries(BUNDLE_DIR)
for entry in source_entries:
    sys.path.insert(0, str(entry))
pth_path = Path(sysconfig.get_paths()["purelib"]) / "taaf_kaggle_sources.pth"
pth_path.write_text("".join(f"{entry}\n" for entry in source_entries))
print(f"taaf.kaggle: wrote {pth_path} ({len(source_entries)} source roots)")

# Solver setup commands (wheels, vLLM server startup, ...) run before the benchmark loads.
def _wm_setup():
    env = _command_env()
    for command in json.loads((BUNDLE_DIR / "setup_commands.json").read_text()):
        print(f"taaf.kaggle: setup command: {command}", flush=True)
        subprocess.run(command, shell=True, check=True, cwd=WORKING_DIR, env=env)
        env = _command_env()
        os.environ.update(env)


# WMPROBE: кэш внимания 8 ГиБ; если сервер не поднялся — боевые 5 ГиБ.
os.environ["TAAF_VLLM_KV_CACHE_MEMORY_BYTES"] = str(8 * 1024 ** 3)
try:
    _wm_setup()
    print("WMPROBE: сервер поднят с кэшем 8 ГиБ", flush=True)
except Exception as _exc:
    print("WMPROBE: с кэшем 8 ГиБ не поднялся (%r) — повтор с 5 ГиБ" % (_exc,), flush=True)
    os.environ["TAAF_VLLM_KV_CACHE_MEMORY_BYTES"] = "5368709120"
    _wm_setup()
    print("WMPROBE: сервер поднят с кэшем 5 ГиБ", flush=True)

# Honour any PYTHONPATH a setup command exported.
for entry in reversed([e for e in os.environ.get("PYTHONPATH", "").split(os.pathsep) if e]):
    if entry not in sys.path:
        sys.path.insert(0, entry)

In [ ]:

# =====================================================================
# WMPROBE: зонд модели мира v3 против своего vLLM (два плеча по очереди)
# =====================================================================
import json, os, subprocess, sys, time
from pathlib import Path
_W = Path("/kaggle/working/wm"); (_W / "scripts").mkdir(parents=True, exist_ok=True)
(_W / "runs/flash_v1_phaseA").mkdir(parents=True, exist_ok=True)
(_W / "scripts/wm_cegis_probe.py").write_text('"""Зонд: пишет ли НАША модель рабочий симулятор игры — цикл Twin без квоты Kaggle (26.09).\n\nМодель та же (Qwen3.8-Flash-Next через HF Inference, поставщик featherless-ai), рассуждение включено.\nДля каждой игры: переходы первого уровня из записанного прогона (runs/flash_v1_phaseA), воспроизведённые на\nлокальном движке. Первые TRAIN переходов даются модели (начальная доска целиком + по каждому ходу изменившиеся\nклетки); модель пишет `def step(grid, action)`. Проверка — в настоящей песочнице v3 (тот же белый список, что в бою).\nЕсли не все переходы воспроизведены — контрпримеры и до REPAIRS кругов ремонта (цикл CEGIS из статьи Twin).\nЗатем обобщение: отложенные переходы того же уровня, которых модель не видела.\n\nМетрики: игр с точным воспроизведением обучающих переходов; доля точно предсказанных отложенных переходов.\nПорог, записанный ДО запуска: если точное воспроизведение меньше чем в трети игр — цикл Twin нашей модели не по силам,\nи прогон arc3-nextfork-twin читать только как проверку, а не как надежду.\nusage: .venv/bin/python scripts/wm_cegis_probe.py [--games all] [--train 16] [--repairs 3]\n"""\nfrom __future__ import annotations\nimport argparse, json, logging, os, re, sys, time\nfrom concurrent.futures import ThreadPoolExecutor\nfrom pathlib import Path\nimport numpy as np\nimport requests\n\nROOT = Path(__file__).resolve().parents[1]\nsys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "vendor/ARC-AGI-3-Agents"))\nsys.path.insert(0, os.environ.get("WM_INFER_SRC", "/tmp/nf_kaggle/src/ARC3-Inference"))\nlogging.disable(logging.ERROR)\nimport arc_agi\nfrom arc_agi import OperationMode\nfrom arcengine import GameAction\nfrom inference.agent.python_tool_sandbox import run_sandboxed_python\n\n# свой сервер на поде: WM_URL=http://127.0.0.1:1234/v1/chat/completions WM_MODEL=Qwen/Qwen3.8-Flash-Next-NVFP4\nURL = os.environ.get("WM_URL", "https://router.huggingface.co/v1/chat/completions")\nMODEL = os.environ.get("WM_MODEL", "Qwen/Qwen3.8-Flash-Next:featherless-ai")\nMAX_TOKENS = int(os.environ.get("WM_MAX_TOKENS", "32000"))\nHEX = "0123456789abcdef"\nNAMES = {"ACTION1": "UP", "ACTION2": "DOWN", "ACTION3": "LEFT", "ACTION4": "RIGHT", "ACTION5": "SPACE", "ACTION7": "ACTION7"}\n\n\ndef env_token():\n    if os.environ.get("WM_URL"):\n        return "local"\n    for line in open(ROOT / ".env", encoding="utf-8"):\n        if line.startswith("HF_INFERENCE_TOKEN="):\n            return line.split("=", 1)[1].strip()\n    raise SystemExit("нет HF_INFERENCE_TOKEN в .env")\n\n\ndef act_str(a):\n    if a["id"] == "ACTION6":\n        d = a.get("data") or {}\n        return "MOUSE(row=%d, col=%d)" % (d.get("y"), d.get("x"))\n    return NAMES.get(a["id"], a["id"])\n\n\ndef level1_transitions(game_id, history):\n    arc = arc_agi.Arcade(operation_mode=getattr(OperationMode, os.environ.get("WM_MODE", "NORMAL")), environments_dir=os.environ.get("WM_ENV_DIR", str(ROOT / "environment_files")))\n    env = arc.make(game_id); fr = env.reset()\n    g = np.asarray(fr.frame[-1]).tolist(); lv = getattr(fr, "levels_completed", 0) or 0\n    out = []\n    for rec in history:\n        a = rec.get("action") or {}\n        if not a.get("id") or a["id"] == "RESET":\n            continue\n        fr2 = env.step(GameAction[a["id"]], data=a.get("data"))\n        if fr2 is None or not fr2.frame:\n            break\n        lv2 = getattr(fr2, "levels_completed", lv) or lv\n        if lv2 != lv:\n            break\n        g2 = np.asarray(fr2.frame[-1]).tolist()\n        if len(g2) != len(g):\n            break\n        out.append((g, act_str(a), g2)); g = g2\n    return out\n\n\ndef hud_lines(train):\n    """Линии полосы у края (3 клетки), меняющиеся в >=50% обучающих переходов: счётчики ходов, полосы энергии."""\n    H, W = len(train[0][0]), len(train[0][0][0]); rows = {}; cols = {}\n    for b, a, af in train:\n        rr = set(); cc = set()\n        for r in range(H):\n            for c in range(W):\n                if b[r][c] != af[r][c]:\n                    if r <= 2 or r >= H - 3: rr.add(r)\n                    if c <= 2 or c >= W - 3: cc.add(c)\n        for r in rr: rows[r] = rows.get(r, 0) + 1\n        for c in cc: cols[c] = cols.get(c, 0) + 1\n    need = 0.5 * len(train)\n    return sorted(r for r, k in rows.items() if k >= need), sorted(c for c, k in cols.items() if k >= need)\n\n\ndef board_text(g):\n    return "\\n".join("".join(HEX[v] for v in row) for row in g)\n\n\ndef diff_text(b, a):\n    """Изменения — отрезками по строке: (r,c1-c2):старый>новый. Полную доску не вставляем: окно 32k (26.09)."""\n    cells = [(r, c, b[r][c], a[r][c]) for r in range(len(b)) for c in range(len(b[0])) if b[r][c] != a[r][c]]\n    if not cells:\n        return "no change"\n    runs = []\n    for r, c, x, y in cells:\n        if runs and runs[-1][0] == r and runs[-1][2] == c - 1 and runs[-1][3] == x and runs[-1][4] == y:\n            runs[-1][2] = c\n        else:\n            runs.append([r, c, c, x, y])\n    parts = ["(%d,%s):%s>%s" % (r, str(c1) if c1 == c2 else "%d-%d" % (c1, c2), HEX[x], HEX[y]) for r, c1, c2, x, y in runs]\n    more = ""\n    if len(parts) > 120:\n        more = " ... (+%d more runs)" % (len(parts) - 120); parts = parts[:120]\n    return "%d cells changed: %s%s" % (len(cells), " ".join(parts), more)\n\n\ndef prompt(trans, hud=None):\n    lines = [\n        "You are reverse-engineering a deterministic grid game from observations.",\n        "The board is a 64x64 grid of colors 0-15 (shown as hex digits, one row per line, row 0 at top).",\n        "Actions: UP, DOWN, LEFT, RIGHT, SPACE, ACTION7, or MOUSE(row=R, col=C) (a click on a cell).",\n        "Below: the starting board, then consecutive transitions; each starts from the previous result.",\n        "Write a Python function `def step(grid, action):` that takes the board as a list of lists of ints and the",\n        "action string, and returns the next board (list of lists of ints). It must reproduce EVERY transition exactly.",\n        "Model objects and rules (what moves, what blocks, what a click does), not a lookup table of these transitions.",\n        "Only builtins and the modules collections/itertools/copy are available. Reply with one ```python block.",\n        "Keep your analysis short (well under 3000 words): form a hypothesis, then write the code; the checker will send counterexamples.",\n        "", "STARTING BOARD:", board_text(trans[0][0]), "", "TRANSITIONS:"]\n    if hud and (hud[0] or hud[1]):\n        lines[-4:-4] = ["Rows %s and columns %s at the board edge look like a status indicator (step counter / energy bar)."\n                        " Your step may leave those cells unchanged: the checker ignores them. Focus on the playfield." % (hud[0], hud[1])]\n    for i, (b, a, af) in enumerate(trans):\n        lines.append("#%d %s -> %s" % (i, a, diff_text(b, af)))\n    return "\\n".join(lines)\n\n\ndef ask(token, messages, tries=3, think=True):\n    budget = max(2000, min(MAX_TOKENS, 32000 - len(json.dumps(messages)) * 9 // 10))\n    body = {"model": MODEL, "messages": messages, "max_tokens": budget, "temperature": 0.6, "top_p": 0.95}\n    if not think:\n        body["chat_template_kwargs"] = {"enable_thinking": False}\n    for k in range(tries):\n        try:\n            r = requests.post(URL, headers={"Authorization": "Bearer " + token, "Content-Type": "application/json"},\n                              json=body, timeout=900).json()\n            if "choices" in r:\n                ch = r["choices"][0]; m = ch["message"]; u = dict(r.get("usage") or {}); u["finish"] = ch.get("finish_reason")\n                return (m.get("content") or ""), u\n            err = json.dumps(r)[:200]\n        except Exception as e:\n            err = repr(e)\n        time.sleep(10 * (k + 1))\n    return "", {"error": err}\n\n\ndef extract_code(text):\n    m = re.findall(r"```(?:python)?\\n(.*?)```", text, re.S)\n    return max(m, key=len) if m else ""\n\n\nVALIDATE = r\'\'\'\nimport json as _j\n_T = __TRANS__\n_MR, _MC = __MASK__\ndef _eq(p, q):\n    if p == q: return True\n    if not _MR and not _MC: return False\n    try:\n        return all(p[r][c] == q[r][c] for r in range(len(q)) for c in range(len(q[0])) if r not in _MR and c not in _MC)\n    except Exception:\n        return False\n_res = {"ok": 0, "n": len(_T), "cex": []}\nfor _i, (_b, _a, _af) in enumerate(_T):\n    try:\n        _p = step([r[:] for r in _b], _a)\n        _good = _eq(_p, _af)\n    except Exception as _e:\n        _good = False; _p = None\n        if len(_res["cex"]) < 3: _res["cex"].append("#%d %s: step raised %r" % (_i, _a, _e))\n    if _good:\n        _res["ok"] += 1\n    elif _p is not None and len(_res["cex"]) < 3:\n        _d = [(r, c, _p[r][c], _af[r][c]) for r in range(min(len(_p), len(_af))) for c in range(min(len(_p[r]), len(_af[r]))) if _p[r][c] != _af[r][c] and r not in _MR and c not in _MC]\n        _res["cex"].append("#%d %s: %d cells wrong, e.g. %s (row,col,predicted,observed)" % (_i, _a, len(_d), _d[:8]))\nprint("RESULT" + _j.dumps(_res))\n\'\'\'\n\n\ndef validate(code, trans, mask=((), ())):\n    src = code + "\\n" + VALIDATE.replace("__TRANS__", json.dumps([[b, a, af] for b, a, af in trans])).replace(\n        "__MASK__", json.dumps([list(mask[0]), list(mask[1])]))\n    out = run_sandboxed_python(code=src, timeout_seconds=30, initial_state={"current_frame": None, "history": []},\n                               action_handler=lambda a: {})\n    m = re.search(r"RESULT(\\{.*\\})", out.get("stdout", ""))\n    if not m:\n        return {"ok": 0, "n": len(trans), "cex": ["code failed: %s" % (out.get("error") or "no result")[:300]]}\n    return json.loads(m.group(1))\n\n\ndef run_game(args):\n    gid, history, token, n_train, repairs = args\n    trans = level1_transitions(gid, history)\n    if len(trans) < 6:\n        return {"game": gid[:4], "skip": "мало переходов (%d)" % len(trans)}\n    train, test = trans[:n_train], trans[n_train:n_train + 16]\n    mask = hud_lines(train) if os.environ.get("WM_MASK_HUD") == "1" else ([], [])\n    msgs = [{"role": "user", "content": prompt(train, mask)}]\n    hist = []; tokens = 0; best = (-1, ""); texts = []\n    for rnd in range(repairs + 1):\n        text, usage = ask(token, msgs, think=os.environ.get("WM_THINK", "1") == "1")\n        tokens += int(usage.get("total_tokens", 0) or 0)\n        code = extract_code(text)\n        if not code and "error" not in usage:   # рассуждение съело потолок: просим только код, без рассуждения\n            text, u2 = ask(token, msgs + [{"role": "user", "content": "Your analysis ran out of space. Reply now with ONLY the ```python block implementing step, no analysis."}], think=False)\n            tokens += int(u2.get("total_tokens", 0) or 0); code = extract_code(text)\n            hist.append({"round": rnd, "rescue": True, "finish": usage.get("finish"), "got_code": bool(code)})\n        texts.append(text)\n        if not code:\n            hist.append({"round": rnd, "ok": 0, "note": "нет кода " + str(usage.get("error", ""))[:80]}); break\n        v = validate(code, train, mask)\n        hist.append({"round": rnd, "ok": v["ok"], "n": v["n"], "cex": v["cex"][:2]})\n        if v["ok"] > best[0]:\n            best = (v["ok"], code)\n        if v["ok"] == v["n"]:\n            break\n        msgs += [{"role": "assistant", "content": text},\n                 {"role": "user", "content": "Your step reproduces %d of %d transitions. Counterexamples:\\n%s\\nFix step and reply with the full corrected ```python block." % (v["ok"], v["n"], "\\n".join(v["cex"]))}]\n    code = best[1]\n    ident = "def step(grid, action):\\n    return grid"\n    base = validate(ident, test, mask) if test else {"ok": 0}   # нулевой базлайн: «ничего не меняй»\n    exact = best[0] == len(train)\n    gen = validate(code, test, mask) if (code and test) else {"ok": 0, "n": 0}\n    full = validate(code, train) if (code and (mask[0] or mask[1])) else None   # точность вместе с индикатором\n    return {"game": gid[:4], "train": len(train), "rounds": len(hist), "train_ok": max(best[0], 0),\n            "exact": bool(exact), "test_ok": gen["ok"], "test_n": gen["n"], "test_identity_ok": base["ok"], "tokens": tokens,\n            "mask": mask, "train_ok_with_hud": (full or {}).get("ok"), "hist": hist, "code": code, "texts": texts}\n\n\ndef main():\n    ap = argparse.ArgumentParser(); ap.add_argument("--games", default="all"); ap.add_argument("--train", type=int, default=16)\n    ap.add_argument("--repairs", type=int, default=3); ap.add_argument("--par", type=int, default=4); a = ap.parse_args()\n    token = env_token()\n    bench = json.loads((ROOT / "runs/flash_v1_phaseA/benchmark.json").read_text())["game_runs"]\n    want = None if a.games == "all" else set(a.games.split(","))\n    jobs = [(g["game_id"], g.get("history") or [], token, a.train, a.repairs) for g in bench if want is None or g["game_id"][:4] in want]\n    t0 = time.time(); rows = []\n    with ThreadPoolExecutor(a.par) as ex:\n        for r in ex.map(run_game, jobs):\n            rows.append(r)\n            print(json.dumps({k: v for k, v in r.items() if k not in ("hist", "code", "texts")}, ensure_ascii=False), flush=True)\n    Path(os.environ.get("WM_OUT", str(ROOT / "runs/wm_cegis_probe.json"))).write_text(json.dumps(rows, ensure_ascii=False, indent=1))\n    done = [r for r in rows if "skip" not in r]\n    ex_ = sum(r["exact"] for r in done); r0 = sum(1 for r in done if [h for h in r["hist"] if "ok" in h][:1] and [h for h in r["hist"] if "ok" in h][0]["ok"] == r["train"])\n    tn = sum(r["test_n"] for r in done); tk = sum(r["test_ok"] for r in done)\n    ti = sum(r.get("test_identity_ok", 0) for r in done)\n    beat = sum(1 for r in done if r["test_ok"] > r.get("test_identity_ok", 0))\n    print("базлайн «ничего не меняй» на отложенных: %d | модель лучше базлайна в %d играх" % (ti, beat))\n    print("\\nигр: %d (пропущено %d) | точно с первой попытки: %d | точно после ремонта: %d (%.0f%%) | отложенные переходы: %d/%d (%.0f%%) | токенов %d | %.0f мин"\n          % (len(done), len(rows) - len(done), r0, ex_, 100 * ex_ / max(len(done), 1), tk, tn, 100 * tk / max(tn, 1),\n             sum(r["tokens"] for r in done), (time.time() - t0) / 60))\n    print("ПОРОГ: точно после ремонта в трети игр и больше ->", "ДОСТИГНУТ" if ex_ * 3 >= len(done) else "НЕ достигнут")\n\n\nif __name__ == "__main__":\n    main()\n')
(_W / "runs/flash_v1_phaseA/benchmark.json").write_text('{"game_runs": [{"game_id": "tn36-ef4dde99", "history": [{"action": {"id": "ACTION6", "data": {"x": 36, "y": 55}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 14}}}, {"action": {"id": "ACTION6", "data": {"x": 36, "y": 55}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 35}}}, {"action": {"id": "ACTION6", "data": {"x": 21, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 19, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 21, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 26, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 36, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 41, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 14}}}, {"action": {"id": "ACTION6", "data": {"x": 26, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 36, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 49}}}, {"action": {"id": "ACTION6", "data": {"x": 36, "y": 55}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 34}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 12}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 17}}}, {"action": {"id": "ACTION6", "data": {"x": 29, "y": 14}}}, {"action": {"id": "ACTION6", "data": {"x": 34, "y": 14}}}, {"action": {"id": "ACTION6", "data": {"x": 32, "y": 34}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 35}}}, {"action": {"id": "ACTION6", "data": {"x": 32, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 38}}}, {"action": {"id": "ACTION6", "data": {"x": 21, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 41, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 13}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 13}}}, {"action": {"id": "ACTION6", "data": {"x": 32, "y": 13}}}, {"action": {"id": "ACTION6", "data": {"x": 33, "y": 13}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 14}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 14}}}, {"action": {"id": "ACTION6", "data": {"x": 32, "y": 14}}}, {"action": {"id": "ACTION6", "data": {"x": 33, "y": 14}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 15}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 15}}}, {"action": {"id": "ACTION6", "data": {"x": 32, "y": 15}}}, {"action": {"id": "ACTION6", "data": {"x": 33, "y": 15}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 16}}}, {"action": {"id": "ACTION6", "data": {"x": 33, "y": 16}}}, {"action": {"id": "ACTION6", "data": {"x": 29, "y": 33}}}, {"action": {"id": "ACTION6", "data": {"x": 34, "y": 33}}}, {"action": {"id": "ACTION6", "data": {"x": 29, "y": 34}}}, {"action": {"id": "ACTION6", "data": {"x": 34, "y": 34}}}, {"action": {"id": "ACTION6", "data": {"x": 29, "y": 35}}}, {"action": {"id": "ACTION6", "data": {"x": 34, "y": 35}}}, {"action": {"id": "ACTION6", "data": {"x": 29, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 32, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 34, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 29, "y": 37}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 37}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 37}}}, {"action": {"id": "ACTION6", "data": {"x": 32, "y": 37}}}, {"action": {"id": "ACTION6", "data": {"x": 33, "y": 37}}}, {"action": {"id": "ACTION6", "data": {"x": 34, "y": 37}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 21, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 26, "y": 44}}}, {"action": {"id": "RESET", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 41, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 36, "y": 55}}}, {"action": {"id": "ACTION6", "data": {"x": 20, "y": 30}}}, {"action": {"id": "ACTION6", "data": {"x": 21, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 41, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 21, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 26, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 36, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 41, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 29, "y": 13}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 13}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 13}}}, {"action": {"id": "ACTION6", "data": {"x": 32, "y": 13}}}, {"action": {"id": "ACTION6", "data": {"x": 33, "y": 13}}}, {"action": {"id": "ACTION6", "data": {"x": 29, "y": 14}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 14}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 14}}}, {"action": {"id": "ACTION6", "data": {"x": 32, "y": 14}}}, {"action": {"id": "ACTION6", "data": {"x": 33, "y": 14}}}, {"action": {"id": "ACTION6", "data": {"x": 29, "y": 15}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 15}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 15}}}, {"action": {"id": "ACTION6", "data": {"x": 32, "y": 15}}}, {"action": {"id": "ACTION6", "data": {"x": 33, "y": 15}}}, {"action": {"id": "ACTION6", "data": {"x": 29, "y": 33}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 33}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 33}}}, {"action": {"id": "ACTION6", "data": {"x": 32, "y": 33}}}, {"action": {"id": "ACTION6", "data": {"x": 33, "y": 33}}}, {"action": {"id": "ACTION6", "data": {"x": 29, "y": 34}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 34}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 34}}}, {"action": {"id": "ACTION6", "data": {"x": 32, "y": 34}}}, {"action": {"id": "ACTION6", "data": {"x": 33, "y": 34}}}, {"action": {"id": "ACTION6", "data": {"x": 29, "y": 35}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 35}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 35}}}, {"action": {"id": "ACTION6", "data": {"x": 32, "y": 35}}}, {"action": {"id": "ACTION6", "data": {"x": 33, "y": 35}}}, {"action": {"id": "ACTION6", "data": {"x": 21, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 26, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 36, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 36, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 41, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 26, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 36, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 21, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 26, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 21, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 41, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 26, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 36, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 41, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 21, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 26, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 44}}}, {"action": {"id": "RESET", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 36, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 21, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 41, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 26, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 36, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 41, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 36, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 41, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 21, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 41, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 36, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 21, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 26, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 21, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 36, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "RESET", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 14}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 16}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 19}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 24}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 26}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 29}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 32}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 34}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 34}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 32}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 29}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 26}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 24}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 19}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 16}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 14}}}, {"action": {"id": "ACTION6", "data": {"x": 36, "y": 55}}}, {"action": {"id": "ACTION6", "data": {"x": 35, "y": 52}}}, {"action": {"id": "ACTION6", "data": {"x": 35, "y": 50}}}, {"action": {"id": "ACTION6", "data": {"x": 34, "y": 47}}}, {"action": {"id": "ACTION6", "data": {"x": 34, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 33, "y": 42}}}, {"action": {"id": "ACTION6", "data": {"x": 32, "y": 39}}}, {"action": {"id": "ACTION6", "data": {"x": 32, "y": 37}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 34}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 14}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 18}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 26}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 30}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 33}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 37}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 41}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 45}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 21, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 41, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 41, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 21, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 21, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 26, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 36, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 41, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 41, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 36, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 26, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 21, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 21, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 41, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 26, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 36, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 36, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 26, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "RESET", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 21, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 41, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 36, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 41, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 41, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 36, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 41, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 26, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 41, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 36, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 41, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 41, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 36, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 41, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 21, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 41, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 36, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 41, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 41, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 36, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 41, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 26, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 41, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 36, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 41, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 41, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 36, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 41, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 14}}}, {"action": {"id": "ACTION6", "data": {"x": 21, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 14}}}, {"action": {"id": "ACTION6", "data": {"x": 26, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 14}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 14}}}, {"action": {"id": "ACTION6", "data": {"x": 36, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 14}}}, {"action": {"id": "ACTION6", "data": {"x": 41, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 34}}}, {"action": {"id": "ACTION6", "data": {"x": 21, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 34}}}, {"action": {"id": "ACTION6", "data": {"x": 26, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 34}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 34}}}, {"action": {"id": "ACTION6", "data": {"x": 36, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 34}}}, {"action": {"id": "ACTION6", "data": {"x": 41, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 21, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 36, "y": 55}}}, {"action": {"id": "ACTION6", "data": {"x": 21, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 26, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 36, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 41, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "RESET", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 36, "y": 55}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 34}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 33}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 14}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 13}}}, {"action": {"id": "ACTION6", "data": {"x": 39, "y": 42}}}, {"action": {"id": "ACTION6", "data": {"x": 39, "y": 43}}}, {"action": {"id": "ACTION6", "data": {"x": 39, "y": 47}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 48}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 49}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 50}}}, {"action": {"id": "ACTION6", "data": {"x": 23, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 34, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 17, "y": 6}}}, {"action": {"id": "ACTION6", "data": {"x": 24, "y": 20}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 35}}}, {"action": {"id": "ACTION6", "data": {"x": 20, "y": 41}}}, {"action": {"id": "ACTION6", "data": {"x": 32, "y": 3}}}, {"action": {"id": "ACTION6", "data": {"x": 32, "y": 63}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 42}}}, {"action": {"id": "RESET", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 13}}}, {"action": {"id": "ACTION6", "data": {"x": 32, "y": 13}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 14}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 33}}}, {"action": {"id": "ACTION6", "data": {"x": 32, "y": 33}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 34}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 35}}}, {"action": {"id": "ACTION6", "data": {"x": 5, "y": 1}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 1}}}, {"action": {"id": "ACTION6", "data": {"x": 55, "y": 1}}}, {"action": {"id": "ACTION6", "data": {"x": 32, "y": 2}}}, {"action": {"id": "ACTION6", "data": {"x": 32, "y": 5}}}, {"action": {"id": "ACTION6", "data": {"x": 32, "y": 48}}}, {"action": {"id": "ACTION6", "data": {"x": 32, "y": 60}}}, {"action": {"id": "ACTION6", "data": {"x": 32, "y": 62}}}, {"action": {"id": "ACTION6", "data": {"x": 21, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 26, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 36, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 21, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 36, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 21, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 26, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 36, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 41, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 21, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 26, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 26, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 36, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 21, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 41, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 21, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 21, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 26, "y": 44}}}, {"action": {"id": "RESET", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 21, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 36, "y": 55}}}, {"action": {"id": "ACTION6", "data": {"x": 21, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 36, "y": 55}}}, {"action": {"id": "ACTION6", "data": {"x": 21, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 26, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 36, "y": 55}}}, {"action": {"id": "ACTION6", "data": {"x": 21, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 36, "y": 55}}}, {"action": {"id": "ACTION6", "data": {"x": 21, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 26, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 36, "y": 55}}}, {"action": {"id": "ACTION6", "data": {"x": 21, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 36, "y": 55}}}, {"action": {"id": "ACTION6", "data": {"x": 21, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 26, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 36, "y": 55}}}, {"action": {"id": "ACTION6", "data": {"x": 21, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 36, "y": 55}}}, {"action": {"id": "ACTION6", "data": {"x": 21, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 26, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 36, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 36, "y": 55}}}, {"action": {"id": "ACTION6", "data": {"x": 21, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 36, "y": 55}}}, {"action": {"id": "ACTION6", "data": {"x": 21, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 26, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 36, "y": 55}}}, {"action": {"id": "ACTION6", "data": {"x": 21, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 36, "y": 55}}}, {"action": {"id": "ACTION6", "data": {"x": 33, "y": 55}}}, {"action": {"id": "ACTION6", "data": {"x": 36, "y": 52}}}, {"action": {"id": "ACTION6", "data": {"x": 36, "y": 57}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 55}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 26, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 44}}}, {"action": {"id": "RESET", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "RESET", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 0, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 21, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 44}}}]}, {"game_id": "lf52-271a04aa", "history": [{"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 19, "y": 19}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 19}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 19}}}, {"action": {"id": "ACTION6", "data": {"x": 42, "y": 19}}}, {"action": {"id": "ACTION6", "data": {"x": 42, "y": 19}}}, {"action": {"id": "ACTION6", "data": {"x": 42, "y": 31}}}, {"action": {"id": "ACTION6", "data": {"x": 42, "y": 37}}}, {"action": {"id": "ACTION6", "data": {"x": 42, "y": 25}}}, {"action": {"id": "ACTION6", "data": {"x": 14, "y": 16}}}, {"action": {"id": "ACTION6", "data": {"x": 26, "y": 16}}}, {"action": {"id": "ACTION6", "data": {"x": 26, "y": 16}}}, {"action": {"id": "ACTION6", "data": {"x": 38, "y": 16}}}, {"action": {"id": "ACTION6", "data": {"x": 44, "y": 16}}}, {"action": {"id": "ACTION6", "data": {"x": 32, "y": 16}}}, {"action": {"id": "ACTION6", "data": {"x": 6, "y": 56}}}, {"action": {"id": "ACTION6", "data": {"x": 14, "y": 16}}}, {"action": {"id": "ACTION6", "data": {"x": 26, "y": 16}}}, {"action": {"id": "ACTION6", "data": {"x": 26, "y": 16}}}, {"action": {"id": "ACTION6", "data": {"x": 38, "y": 16}}}, {"action": {"id": "ACTION6", "data": {"x": 44, "y": 16}}}, {"action": {"id": "ACTION6", "data": {"x": 32, "y": 16}}}, {"action": {"id": "ACTION6", "data": {"x": 32, "y": 34}}}, {"action": {"id": "ACTION6", "data": {"x": 33, "y": 16}}}, {"action": {"id": "ACTION6", "data": {"x": 4, "y": 53}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 6, "y": 56}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 45, "y": 15}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 45, "y": 15}}}, {"action": {"id": "ACTION6", "data": {"x": 39, "y": 15}}}, {"action": {"id": "ACTION6", "data": {"x": 33, "y": 15}}}, {"action": {"id": "ACTION6", "data": {"x": 45, "y": 15}}}, {"action": {"id": "ACTION6", "data": {"x": 33, "y": 15}}}, {"action": {"id": "ACTION6", "data": {"x": 21, "y": 15}}}, {"action": {"id": "ACTION6", "data": {"x": 15, "y": 15}}}, {"action": {"id": "ACTION6", "data": {"x": 21, "y": 15}}}, {"action": {"id": "ACTION6", "data": {"x": 27, "y": 15}}}, {"action": {"id": "ACTION6", "data": {"x": 21, "y": 15}}}, {"action": {"id": "ACTION6", "data": {"x": 9, "y": 15}}}, {"action": {"id": "ACTION6", "data": {"x": 45, "y": 15}}}, {"action": {"id": "ACTION6", "data": {"x": 21, "y": 15}}}, {"action": {"id": "ACTION6", "data": {"x": 9, "y": 15}}}, {"action": {"id": "ACTION6", "data": {"x": 33, "y": 15}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 6, "y": 57}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 44, "y": 51}}}, {"action": {"id": "ACTION6", "data": {"x": 37, "y": 50}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 38, "y": 15}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 44, "y": 15}}}, {"action": {"id": "ACTION6", "data": {"x": 49, "y": 15}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 49, "y": 15}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 4, "y": 52}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 50, "y": 15}}}, {"action": {"id": "ACTION6", "data": {"x": 14, "y": 15}}}, {"action": {"id": "ACTION6", "data": {"x": 26, "y": 15}}}, {"action": {"id": "ACTION6", "data": {"x": 50, "y": 21}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 50, "y": 21}}}]}, {"game_id": "cn04-2fe56bfb", "history": [{"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}]}, {"game_id": "bp35-0a0ad940", "history": [{"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 20, "y": 42}}}, {"action": {"id": "ACTION6", "data": {"x": 33, "y": 33}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 27, "y": 33}}}, {"action": {"id": "ACTION6", "data": {"x": 27, "y": 39}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 21, "y": 33}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 33, "y": 33}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 21, "y": 33}}}, {"action": {"id": "ACTION6", "data": {"x": 21, "y": 33}}}, {"action": {"id": "RESET", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 45, "y": 32}}}, {"action": {"id": "ACTION6", "data": {"x": 45, "y": 27}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 50, "y": 33}}}, {"action": {"id": "RESET", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 45, "y": 33}}}, {"action": {"id": "ACTION6", "data": {"x": 45, "y": 33}}}, {"action": {"id": "ACTION6", "data": {"x": 45, "y": 33}}}, {"action": {"id": "RESET", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 39, "y": 17}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 21, "y": 33}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 15, "y": 33}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 23, "y": 33}}}, {"action": {"id": "RESET", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 37, "y": 33}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 49, "y": 33}}}, {"action": {"id": "ACTION6", "data": {"x": 39, "y": 33}}}]}, {"game_id": "wa30-ee6fef47", "history": [{"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "RESET", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "RESET", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "RESET", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "RESET", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}]}, {"game_id": "lp85-305b61c3", "history": [{"action": {"id": "ACTION6", "data": {"x": 13, "y": 20}}}, {"action": {"id": "ACTION6", "data": {"x": 5, "y": 32}}}, {"action": {"id": "ACTION6", "data": {"x": 58, "y": 32}}}, {"action": {"id": "ACTION6", "data": {"x": 5, "y": 32}}}, {"action": {"id": "ACTION6", "data": {"x": 5, "y": 32}}}, {"action": {"id": "ACTION6", "data": {"x": 5, "y": 32}}}, {"action": {"id": "ACTION6", "data": {"x": 5, "y": 32}}}, {"action": {"id": "ACTION6", "data": {"x": 5, "y": 32}}}, {"action": {"id": "ACTION6", "data": {"x": 20, "y": 17}}}, {"action": {"id": "ACTION6", "data": {"x": 39, "y": 17}}}, {"action": {"id": "ACTION6", "data": {"x": 14, "y": 26}}}, {"action": {"id": "ACTION6", "data": {"x": 48, "y": 26}}}, {"action": {"id": "ACTION6", "data": {"x": 20, "y": 17}}}]}, {"game_id": "r11l-495a7899", "history": [{"action": {"id": "ACTION6", "data": {"x": 17, "y": 47}}}, {"action": {"id": "ACTION6", "data": {"x": 15, "y": 15}}}, {"action": {"id": "ACTION6", "data": {"x": 39, "y": 21}}}, {"action": {"id": "ACTION6", "data": {"x": 39, "y": 21}}}, {"action": {"id": "ACTION6", "data": {"x": 27, "y": 59}}}, {"action": {"id": "ACTION6", "data": {"x": 10, "y": 40}}}, {"action": {"id": "ACTION6", "data": {"x": 63, "y": 42}}}, {"action": {"id": "ACTION6", "data": {"x": 39, "y": 21}}}, {"action": {"id": "ACTION6", "data": {"x": 15, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 20, "y": 10}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 6}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 20}}}, {"action": {"id": "ACTION6", "data": {"x": 10, "y": 6}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 30}}}, {"action": {"id": "ACTION6", "data": {"x": 25, "y": 6}}}, {"action": {"id": "ACTION6", "data": {"x": 50, "y": 3}}}, {"action": {"id": "ACTION6", "data": {"x": 10, "y": 45}}}, {"action": {"id": "ACTION6", "data": {"x": 8, "y": 21}}}, {"action": {"id": "ACTION6", "data": {"x": 52, "y": 18}}}, {"action": {"id": "ACTION6", "data": {"x": 45, "y": 35}}}, {"action": {"id": "ACTION6", "data": {"x": 62, "y": 18}}}, {"action": {"id": "ACTION6", "data": {"x": 54, "y": 48}}}, {"action": {"id": "ACTION6", "data": {"x": 37, "y": 57}}}, {"action": {"id": "ACTION6", "data": {"x": 49, "y": 9}}}, {"action": {"id": "ACTION6", "data": {"x": 38, "y": 56}}}, {"action": {"id": "ACTION6", "data": {"x": 50, "y": 3}}}, {"action": {"id": "ACTION6", "data": {"x": 45, "y": 40}}}, {"action": {"id": "ACTION6", "data": {"x": 54, "y": 18}}}, {"action": {"id": "ACTION6", "data": {"x": 52, "y": 18}}}, {"action": {"id": "ACTION6", "data": {"x": 60, "y": 18}}}, {"action": {"id": "ACTION6", "data": {"x": 49, "y": 9}}}, {"action": {"id": "ACTION6", "data": {"x": 39, "y": 57}}}, {"action": {"id": "ACTION6", "data": {"x": 45, "y": 35}}}, {"action": {"id": "ACTION6", "data": {"x": 44, "y": 39}}}, {"action": {"id": "ACTION6", "data": {"x": 52, "y": 18}}}, {"action": {"id": "ACTION6", "data": {"x": 45, "y": 35}}}, {"action": {"id": "ACTION6", "data": {"x": 37, "y": 56}}}, {"action": {"id": "ACTION6", "data": {"x": 49, "y": 9}}}, {"action": {"id": "ACTION6", "data": {"x": 46, "y": 40}}}, {"action": {"id": "ACTION6", "data": {"x": 37, "y": 56}}}, {"action": {"id": "ACTION6", "data": {"x": 45, "y": 35}}}, {"action": {"id": "ACTION6", "data": {"x": 54, "y": 18}}}, {"action": {"id": "ACTION6", "data": {"x": 46, "y": 40}}}, {"action": {"id": "ACTION6", "data": {"x": 54, "y": 18}}}, {"action": {"id": "ACTION6", "data": {"x": 45, "y": 35}}}, {"action": {"id": "ACTION6", "data": {"x": 54, "y": 18}}}, {"action": {"id": "ACTION6", "data": {"x": 45, "y": 35}}}, {"action": {"id": "ACTION6", "data": {"x": 54, "y": 18}}}, {"action": {"id": "ACTION6", "data": {"x": 45, "y": 35}}}, {"action": {"id": "ACTION6", "data": {"x": 37, "y": 56}}}, {"action": {"id": "ACTION6", "data": {"x": 52, "y": 18}}}, {"action": {"id": "ACTION6", "data": {"x": 37, "y": 56}}}, {"action": {"id": "ACTION6", "data": {"x": 52, "y": 18}}}, {"action": {"id": "ACTION6", "data": {"x": 37, "y": 56}}}, {"action": {"id": "ACTION6", "data": {"x": 52, "y": 18}}}, {"action": {"id": "ACTION6", "data": {"x": 41, "y": 57}}}, {"action": {"id": "ACTION6", "data": {"x": 37, "y": 57}}}, {"action": {"id": "ACTION6", "data": {"x": 42, "y": 40}}}, {"action": {"id": "ACTION6", "data": {"x": 45, "y": 35}}}, {"action": {"id": "ACTION6", "data": {"x": 37, "y": 56}}}, {"action": {"id": "ACTION6", "data": {"x": 54, "y": 18}}}, {"action": {"id": "ACTION6", "data": {"x": 46, "y": 40}}}, {"action": {"id": "ACTION6", "data": {"x": 42, "y": 40}}}, {"action": {"id": "ACTION6", "data": {"x": 37, "y": 56}}}, {"action": {"id": "ACTION6", "data": {"x": 46, "y": 40}}}, {"action": {"id": "ACTION6", "data": {"x": 42, "y": 40}}}, {"action": {"id": "ACTION6", "data": {"x": 43, "y": 40}}}, {"action": {"id": "ACTION6", "data": {"x": 42, "y": 40}}}, {"action": {"id": "ACTION6", "data": {"x": 50, "y": 3}}}, {"action": {"id": "ACTION6", "data": {"x": 54, "y": 18}}}, {"action": {"id": "RESET", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 30}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 57}}}, {"action": {"id": "ACTION6", "data": {"x": 52, "y": 18}}}, {"action": {"id": "ACTION6", "data": {"x": 8, "y": 21}}}, {"action": {"id": "ACTION6", "data": {"x": 62, "y": 18}}}, {"action": {"id": "ACTION6", "data": {"x": 49, "y": 9}}}, {"action": {"id": "ACTION6", "data": {"x": 57, "y": 18}}}, {"action": {"id": "ACTION6", "data": {"x": 42, "y": 45}}}, {"action": {"id": "ACTION6", "data": {"x": 8, "y": 21}}}, {"action": {"id": "ACTION6", "data": {"x": 42, "y": 40}}}, {"action": {"id": "ACTION6", "data": {"x": 42, "y": 45}}}, {"action": {"id": "ACTION6", "data": {"x": 57, "y": 18}}}, {"action": {"id": "ACTION6", "data": {"x": 38, "y": 57}}}, {"action": {"id": "ACTION6", "data": {"x": 52, "y": 18}}}, {"action": {"id": "ACTION6", "data": {"x": 54, "y": 48}}}, {"action": {"id": "ACTION6", "data": {"x": 49, "y": 41}}}, {"action": {"id": "ACTION6", "data": {"x": 52, "y": 18}}}, {"action": {"id": "ACTION6", "data": {"x": 62, "y": 18}}}, {"action": {"id": "ACTION6", "data": {"x": 38, "y": 57}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 57, "y": 18}}}, {"action": {"id": "ACTION6", "data": {"x": 51, "y": 52}}}, {"action": {"id": "ACTION6", "data": {"x": 49, "y": 41}}}, {"action": {"id": "ACTION6", "data": {"x": 38, "y": 57}}}, {"action": {"id": "ACTION6", "data": {"x": 38, "y": 56}}}, {"action": {"id": "ACTION6", "data": {"x": 39, "y": 56}}}, {"action": {"id": "ACTION6", "data": {"x": 62, "y": 18}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 44}}}]}, {"game_id": "tu93-0768757b", "history": [{"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "RESET", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "RESET", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "RESET", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "RESET", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "RESET", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}]}, {"game_id": "sp80-589a99af", "history": [{"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "RESET", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 37, "y": 5}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "RESET", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 4, "y": 30}}}, {"action": {"id": "ACTION6", "data": {"x": 60, "y": 20}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "RESET", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "RESET", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "RESET", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "RESET", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "RESET", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 37, "y": 5}}}, {"action": {"id": "ACTION6", "data": {"x": 21, "y": 55}}}, {"action": {"id": "ACTION6", "data": {"x": 16, "y": 16}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "RESET", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "RESET", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 30}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 30}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 30}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 30}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 30}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 30}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 30}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 30}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 30}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 30}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 30}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 30}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 30}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 30}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 30}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 30}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 30}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 30}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 30}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 30}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 30}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 30}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 30}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 30}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 30}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 30}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 30}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 30}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 30}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 30}}}, {"action": {"id": "RESET", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "RESET", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "RESET", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "RESET", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "RESET", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 37, "y": 5}}}, {"action": {"id": "ACTION6", "data": {"x": 37, "y": 2}}}, {"action": {"id": "ACTION6", "data": {"x": 21, "y": 55}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 0}}}, {"action": {"id": "ACTION6", "data": {"x": 12, "y": 16}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "RESET", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "RESET", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "RESET", "data": {}}}]}, {"game_id": "m0r0-492f87ba", "history": [{"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}]}, {"game_id": "vc33-5430563c", "history": [{"action": {"id": "ACTION6", "data": {"x": 48, "y": 46}}}, {"action": {"id": "ACTION6", "data": {"x": 50, "y": 46}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 10}}}, {"action": {"id": "ACTION6", "data": {"x": 61, "y": 25}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 29}}}, {"action": {"id": "ACTION6", "data": {"x": 10, "y": 10}}}, {"action": {"id": "ACTION6", "data": {"x": 38, "y": 29}}}, {"action": {"id": "ACTION6", "data": {"x": 61, "y": 33}}}, {"action": {"id": "ACTION6", "data": {"x": 61, "y": 25}}}, {"action": {"id": "ACTION6", "data": {"x": 61, "y": 25}}}, {"action": {"id": "ACTION6", "data": {"x": 61, "y": 25}}}, {"action": {"id": "ACTION6", "data": {"x": 61, "y": 25}}}, {"action": {"id": "ACTION6", "data": {"x": 61, "y": 33}}}, {"action": {"id": "ACTION6", "data": {"x": 61, "y": 33}}}, {"action": {"id": "ACTION6", "data": {"x": 61, "y": 33}}}, {"action": {"id": "ACTION6", "data": {"x": 61, "y": 33}}}, {"action": {"id": "ACTION6", "data": {"x": 61, "y": 33}}}, {"action": {"id": "ACTION6", "data": {"x": 61, "y": 33}}}, {"action": {"id": "ACTION6", "data": {"x": 1, "y": 17}}}, {"action": {"id": "ACTION6", "data": {"x": 1, "y": 25}}}, {"action": {"id": "ACTION6", "data": {"x": 1, "y": 37}}}, {"action": {"id": "ACTION6", "data": {"x": 1, "y": 45}}}, {"action": {"id": "ACTION6", "data": {"x": 1, "y": 17}}}, {"action": {"id": "ACTION6", "data": {"x": 1, "y": 37}}}, {"action": {"id": "ACTION6", "data": {"x": 1, "y": 45}}}, {"action": {"id": "ACTION6", "data": {"x": 1, "y": 45}}}, {"action": {"id": "ACTION6", "data": {"x": 1, "y": 45}}}, {"action": {"id": "ACTION6", "data": {"x": 1, "y": 45}}}, {"action": {"id": "ACTION6", "data": {"x": 1, "y": 45}}}, {"action": {"id": "ACTION6", "data": {"x": 1, "y": 45}}}, {"action": {"id": "ACTION6", "data": {"x": 1, "y": 25}}}, {"action": {"id": "ACTION6", "data": {"x": 1, "y": 45}}}, {"action": {"id": "ACTION6", "data": {"x": 1, "y": 25}}}, {"action": {"id": "ACTION6", "data": {"x": 1, "y": 45}}}, {"action": {"id": "ACTION6", "data": {"x": 1, "y": 25}}}, {"action": {"id": "ACTION6", "data": {"x": 1, "y": 45}}}, {"action": {"id": "ACTION6", "data": {"x": 12, "y": 56}}}, {"action": {"id": "ACTION6", "data": {"x": 16, "y": 56}}}, {"action": {"id": "ACTION6", "data": {"x": 24, "y": 56}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 56}}}, {"action": {"id": "ACTION6", "data": {"x": 46, "y": 56}}}, {"action": {"id": "ACTION6", "data": {"x": 34, "y": 56}}}, {"action": {"id": "ACTION6", "data": {"x": 46, "y": 56}}}, {"action": {"id": "ACTION6", "data": {"x": 46, "y": 56}}}, {"action": {"id": "ACTION6", "data": {"x": 46, "y": 56}}}, {"action": {"id": "ACTION6", "data": {"x": 46, "y": 56}}}, {"action": {"id": "ACTION6", "data": {"x": 46, "y": 56}}}, {"action": {"id": "ACTION6", "data": {"x": 46, "y": 56}}}, {"action": {"id": "ACTION6", "data": {"x": 46, "y": 56}}}, {"action": {"id": "ACTION6", "data": {"x": 46, "y": 56}}}, {"action": {"id": "ACTION6", "data": {"x": 34, "y": 56}}}, {"action": {"id": "ACTION6", "data": {"x": 34, "y": 56}}}, {"action": {"id": "ACTION6", "data": {"x": 24, "y": 56}}}, {"action": {"id": "ACTION6", "data": {"x": 24, "y": 56}}}, {"action": {"id": "ACTION6", "data": {"x": 24, "y": 56}}}, {"action": {"id": "ACTION6", "data": {"x": 24, "y": 56}}}, {"action": {"id": "ACTION6", "data": {"x": 24, "y": 56}}}, {"action": {"id": "ACTION6", "data": {"x": 12, "y": 56}}}, {"action": {"id": "ACTION6", "data": {"x": 12, "y": 56}}}, {"action": {"id": "ACTION6", "data": {"x": 12, "y": 56}}}, {"action": {"id": "ACTION6", "data": {"x": 12, "y": 56}}}, {"action": {"id": "ACTION6", "data": {"x": 12, "y": 56}}}, {"action": {"id": "ACTION6", "data": {"x": 12, "y": 56}}}, {"action": {"id": "ACTION6", "data": {"x": 34, "y": 56}}}, {"action": {"id": "ACTION6", "data": {"x": 24, "y": 56}}}, {"action": {"id": "ACTION6", "data": {"x": 12, "y": 56}}}, {"action": {"id": "ACTION6", "data": {"x": 34, "y": 56}}}, {"action": {"id": "ACTION6", "data": {"x": 24, "y": 56}}}, {"action": {"id": "ACTION6", "data": {"x": 12, "y": 56}}}, {"action": {"id": "ACTION6", "data": {"x": 24, "y": 56}}}, {"action": {"id": "ACTION6", "data": {"x": 12, "y": 56}}}, {"action": {"id": "ACTION6", "data": {"x": 13, "y": 48}}}, {"action": {"id": "ACTION6", "data": {"x": 15, "y": 62}}}, {"action": {"id": "ACTION6", "data": {"x": 9, "y": 62}}}, {"action": {"id": "ACTION6", "data": {"x": 5, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 51, "y": 62}}}, {"action": {"id": "ACTION6", "data": {"x": 9, "y": 62}}}, {"action": {"id": "ACTION6", "data": {"x": 9, "y": 62}}}, {"action": {"id": "ACTION6", "data": {"x": 9, "y": 62}}}, {"action": {"id": "ACTION6", "data": {"x": 39, "y": 62}}}, {"action": {"id": "ACTION6", "data": {"x": 39, "y": 62}}}, {"action": {"id": "ACTION6", "data": {"x": 39, "y": 62}}}, {"action": {"id": "ACTION6", "data": {"x": 51, "y": 62}}}, {"action": {"id": "ACTION6", "data": {"x": 51, "y": 62}}}, {"action": {"id": "ACTION6", "data": {"x": 51, "y": 62}}}, {"action": {"id": "ACTION6", "data": {"x": 51, "y": 62}}}, {"action": {"id": "ACTION6", "data": {"x": 51, "y": 62}}}, {"action": {"id": "ACTION6", "data": {"x": 39, "y": 62}}}, {"action": {"id": "ACTION6", "data": {"x": 39, "y": 62}}}, {"action": {"id": "ACTION6", "data": {"x": 39, "y": 62}}}, {"action": {"id": "ACTION6", "data": {"x": 39, "y": 62}}}, {"action": {"id": "ACTION6", "data": {"x": 39, "y": 62}}}, {"action": {"id": "ACTION6", "data": {"x": 5, "y": 41}}}, {"action": {"id": "ACTION6", "data": {"x": 35, "y": 35}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 40}}}, {"action": {"id": "ACTION6", "data": {"x": 13, "y": 48}}}, {"action": {"id": "ACTION6", "data": {"x": 10, "y": 62}}}, {"action": {"id": "ACTION6", "data": {"x": 10, "y": 62}}}, {"action": {"id": "ACTION6", "data": {"x": 10, "y": 62}}}, {"action": {"id": "ACTION6", "data": {"x": 10, "y": 62}}}, {"action": {"id": "ACTION6", "data": {"x": 10, "y": 62}}}, {"action": {"id": "ACTION6", "data": {"x": 10, "y": 62}}}, {"action": {"id": "ACTION6", "data": {"x": 10, "y": 62}}}, {"action": {"id": "ACTION6", "data": {"x": 16, "y": 62}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 62}}}, {"action": {"id": "ACTION6", "data": {"x": 46, "y": 62}}}, {"action": {"id": "ACTION6", "data": {"x": 52, "y": 62}}}, {"action": {"id": "ACTION6", "data": {"x": 58, "y": 62}}}, {"action": {"id": "ACTION6", "data": {"x": 16, "y": 62}}}, {"action": {"id": "ACTION6", "data": {"x": 10, "y": 62}}}, {"action": {"id": "ACTION6", "data": {"x": 16, "y": 62}}}, {"action": {"id": "ACTION6", "data": {"x": 10, "y": 62}}}]}, {"game_id": "ar25-0c556536", "history": [{"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 49, "y": 19}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 50, "y": 29}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 19, "y": 38}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 19, "y": 20}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 14, "y": 22}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 61, "y": 38}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 34, "y": 17}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 49, "y": 43}}}, {"action": {"id": "ACTION6", "data": {"x": 43, "y": 37}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}]}, {"game_id": "ka59-38d34dbb", "history": [{"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 35, "y": 31}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 43, "y": 31}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 10, "y": 34}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 34}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 34, "y": 43}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 25, "y": 40}}}, {"action": {"id": "ACTION6", "data": {"x": 8, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 5, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 33}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 42}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 56, "y": 33}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 48}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}]}, {"game_id": "sc25-635fd71a", "history": [{"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 24, "y": 49}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 49}}}, {"action": {"id": "ACTION6", "data": {"x": 24, "y": 49}}}, {"action": {"id": "ACTION6", "data": {"x": 25, "y": 55}}}, {"action": {"id": "ACTION6", "data": {"x": 35, "y": 55}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 60}}}, {"action": {"id": "ACTION6", "data": {"x": 25, "y": 55}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 49}}}, {"action": {"id": "ACTION6", "data": {"x": 35, "y": 55}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 60}}}, {"action": {"id": "ACTION6", "data": {"x": 25, "y": 49}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 49}}}, {"action": {"id": "ACTION6", "data": {"x": 25, "y": 55}}}, {"action": {"id": "ACTION6", "data": {"x": 35, "y": 55}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 60}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 24, "y": 20}}}, {"action": {"id": "ACTION6", "data": {"x": 14, "y": 19}}}, {"action": {"id": "ACTION6", "data": {"x": 25, "y": 49}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 25, "y": 49}}}, {"action": {"id": "ACTION6", "data": {"x": 25, "y": 49}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 49}}}, {"action": {"id": "ACTION6", "data": {"x": 35, "y": 49}}}, {"action": {"id": "ACTION6", "data": {"x": 25, "y": 60}}}, {"action": {"id": "ACTION6", "data": {"x": 35, "y": 60}}}, {"action": {"id": "ACTION6", "data": {"x": 25, "y": 49}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 49}}}, {"action": {"id": "ACTION6", "data": {"x": 25, "y": 49}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 49}}}, {"action": {"id": "ACTION6", "data": {"x": 35, "y": 49}}}, {"action": {"id": "RESET", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 25, "y": 49}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 49}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 55}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 25, "y": 49}}}, {"action": {"id": "ACTION6", "data": {"x": 25, "y": 55}}}, {"action": {"id": "ACTION6", "data": {"x": 25, "y": 60}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 49}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 55}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 60}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 50}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 25, "y": 49}}}, {"action": {"id": "ACTION6", "data": {"x": 25, "y": 55}}}, {"action": {"id": "ACTION6", "data": {"x": 25, "y": 60}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 49}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 50}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 55}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 60}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "RESET", "data": {}}}]}, {"game_id": "sk48-d8078629", "history": [{"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 19, "y": 20}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 22, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 13, "y": 17}}}, {"action": {"id": "ACTION6", "data": {"x": 13, "y": 32}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 3, "y": 3}}}, {"action": {"id": "ACTION6", "data": {"x": 3, "y": 3}}}, {"action": {"id": "ACTION6", "data": {"x": 3, "y": 3}}}, {"action": {"id": "ACTION6", "data": {"x": 3, "y": 3}}}, {"action": {"id": "ACTION6", "data": {"x": 3, "y": 3}}}, {"action": {"id": "ACTION6", "data": {"x": 3, "y": 3}}}, {"action": {"id": "ACTION6", "data": {"x": 3, "y": 3}}}, {"action": {"id": "ACTION6", "data": {"x": 3, "y": 3}}}, {"action": {"id": "ACTION6", "data": {"x": 3, "y": 3}}}, {"action": {"id": "ACTION6", "data": {"x": 3, "y": 3}}}, {"action": {"id": "ACTION6", "data": {"x": 3, "y": 3}}}, {"action": {"id": "ACTION6", "data": {"x": 3, "y": 3}}}, {"action": {"id": "ACTION6", "data": {"x": 3, "y": 3}}}, {"action": {"id": "ACTION6", "data": {"x": 3, "y": 3}}}, {"action": {"id": "ACTION6", "data": {"x": 3, "y": 3}}}, {"action": {"id": "ACTION6", "data": {"x": 3, "y": 3}}}, {"action": {"id": "ACTION6", "data": {"x": 3, "y": 3}}}, {"action": {"id": "ACTION6", "data": {"x": 3, "y": 3}}}, {"action": {"id": "ACTION6", "data": {"x": 3, "y": 3}}}, {"action": {"id": "ACTION6", "data": {"x": 3, "y": 3}}}, {"action": {"id": "ACTION6", "data": {"x": 3, "y": 3}}}, {"action": {"id": "ACTION6", "data": {"x": 3, "y": 3}}}, {"action": {"id": "ACTION6", "data": {"x": 3, "y": 3}}}, {"action": {"id": "ACTION6", "data": {"x": 3, "y": 3}}}, {"action": {"id": "ACTION6", "data": {"x": 3, "y": 3}}}, {"action": {"id": "ACTION6", "data": {"x": 3, "y": 3}}}, {"action": {"id": "ACTION6", "data": {"x": 3, "y": 3}}}, {"action": {"id": "ACTION6", "data": {"x": 3, "y": 3}}}, {"action": {"id": "ACTION6", "data": {"x": 3, "y": 3}}}, {"action": {"id": "ACTION6", "data": {"x": 3, "y": 3}}}, {"action": {"id": "ACTION6", "data": {"x": 3, "y": 3}}}, {"action": {"id": "ACTION6", "data": {"x": 3, "y": 3}}}, {"action": {"id": "ACTION6", "data": {"x": 3, "y": 3}}}, {"action": {"id": "ACTION6", "data": {"x": 3, "y": 3}}}, {"action": {"id": "ACTION6", "data": {"x": 3, "y": 3}}}, {"action": {"id": "ACTION6", "data": {"x": 3, "y": 3}}}, {"action": {"id": "ACTION6", "data": {"x": 3, "y": 3}}}, {"action": {"id": "ACTION6", "data": {"x": 3, "y": 3}}}, {"action": {"id": "ACTION6", "data": {"x": 3, "y": 3}}}, {"action": {"id": "ACTION6", "data": {"x": 3, "y": 3}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 4, "y": 4}}}, {"action": {"id": "ACTION6", "data": {"x": 4, "y": 4}}}, {"action": {"id": "ACTION6", "data": {"x": 4, "y": 4}}}, {"action": {"id": "ACTION6", "data": {"x": 4, "y": 4}}}, {"action": {"id": "ACTION6", "data": {"x": 4, "y": 4}}}, {"action": {"id": "ACTION6", "data": {"x": 4, "y": 4}}}, {"action": {"id": "ACTION6", "data": {"x": 4, "y": 4}}}, {"action": {"id": "ACTION6", "data": {"x": 4, "y": 4}}}, {"action": {"id": "ACTION6", "data": {"x": 4, "y": 4}}}, {"action": {"id": "ACTION6", "data": {"x": 4, "y": 4}}}, {"action": {"id": "ACTION6", "data": {"x": 4, "y": 4}}}, {"action": {"id": "ACTION6", "data": {"x": 4, "y": 4}}}, {"action": {"id": "ACTION6", "data": {"x": 4, "y": 4}}}, {"action": {"id": "ACTION6", "data": {"x": 4, "y": 4}}}, {"action": {"id": "ACTION6", "data": {"x": 4, "y": 4}}}, {"action": {"id": "ACTION6", "data": {"x": 4, "y": 4}}}, {"action": {"id": "ACTION6", "data": {"x": 4, "y": 4}}}, {"action": {"id": "ACTION6", "data": {"x": 4, "y": 4}}}, {"action": {"id": "ACTION6", "data": {"x": 4, "y": 4}}}, {"action": {"id": "ACTION6", "data": {"x": 4, "y": 4}}}, {"action": {"id": "ACTION6", "data": {"x": 4, "y": 4}}}, {"action": {"id": "ACTION6", "data": {"x": 4, "y": 4}}}, {"action": {"id": "ACTION6", "data": {"x": 4, "y": 4}}}, {"action": {"id": "ACTION6", "data": {"x": 4, "y": 4}}}, {"action": {"id": "ACTION6", "data": {"x": 4, "y": 4}}}, {"action": {"id": "ACTION6", "data": {"x": 4, "y": 4}}}, {"action": {"id": "ACTION6", "data": {"x": 4, "y": 4}}}, {"action": {"id": "ACTION6", "data": {"x": 4, "y": 4}}}, {"action": {"id": "ACTION6", "data": {"x": 4, "y": 4}}}, {"action": {"id": "ACTION6", "data": {"x": 4, "y": 4}}}, {"action": {"id": "ACTION6", "data": {"x": 4, "y": 4}}}, {"action": {"id": "ACTION6", "data": {"x": 4, "y": 4}}}, {"action": {"id": "ACTION6", "data": {"x": 4, "y": 4}}}, {"action": {"id": "ACTION6", "data": {"x": 4, "y": 4}}}, {"action": {"id": "ACTION6", "data": {"x": 4, "y": 4}}}, {"action": {"id": "ACTION6", "data": {"x": 4, "y": 4}}}, {"action": {"id": "ACTION6", "data": {"x": 4, "y": 4}}}, {"action": {"id": "ACTION6", "data": {"x": 4, "y": 4}}}, {"action": {"id": "ACTION6", "data": {"x": 4, "y": 4}}}, {"action": {"id": "ACTION6", "data": {"x": 4, "y": 4}}}, {"action": {"id": "ACTION6", "data": {"x": 4, "y": 4}}}, {"action": {"id": "ACTION6", "data": {"x": 4, "y": 4}}}, {"action": {"id": "ACTION6", "data": {"x": 4, "y": 4}}}, {"action": {"id": "ACTION6", "data": {"x": 4, "y": 4}}}, {"action": {"id": "ACTION6", "data": {"x": 4, "y": 4}}}, {"action": {"id": "ACTION6", "data": {"x": 4, "y": 4}}}, {"action": {"id": "ACTION6", "data": {"x": 4, "y": 4}}}, {"action": {"id": "ACTION6", "data": {"x": 4, "y": 4}}}, {"action": {"id": "ACTION6", "data": {"x": 6, "y": 6}}}, {"action": {"id": "ACTION6", "data": {"x": 6, "y": 6}}}, {"action": {"id": "ACTION6", "data": {"x": 6, "y": 6}}}, {"action": {"id": "ACTION6", "data": {"x": 6, "y": 6}}}, {"action": {"id": "ACTION6", "data": {"x": 6, "y": 6}}}, {"action": {"id": "ACTION6", "data": {"x": 6, "y": 6}}}, {"action": {"id": "ACTION6", "data": {"x": 6, "y": 6}}}, {"action": {"id": "ACTION6", "data": {"x": 6, "y": 6}}}, {"action": {"id": "ACTION6", "data": {"x": 6, "y": 6}}}, {"action": {"id": "ACTION6", "data": {"x": 6, "y": 6}}}, {"action": {"id": "ACTION6", "data": {"x": 6, "y": 6}}}, {"action": {"id": "ACTION6", "data": {"x": 6, "y": 6}}}, {"action": {"id": "ACTION6", "data": {"x": 6, "y": 6}}}, {"action": {"id": "ACTION6", "data": {"x": 6, "y": 6}}}, {"action": {"id": "ACTION6", "data": {"x": 6, "y": 6}}}, {"action": {"id": "ACTION6", "data": {"x": 6, "y": 6}}}, {"action": {"id": "ACTION6", "data": {"x": 6, "y": 6}}}, {"action": {"id": "ACTION6", "data": {"x": 6, "y": 6}}}, {"action": {"id": "ACTION6", "data": {"x": 6, "y": 6}}}, {"action": {"id": "ACTION6", "data": {"x": 6, "y": 6}}}, {"action": {"id": "ACTION6", "data": {"x": 6, "y": 6}}}, {"action": {"id": "ACTION6", "data": {"x": 6, "y": 6}}}, {"action": {"id": "ACTION6", "data": {"x": 6, "y": 6}}}, {"action": {"id": "ACTION6", "data": {"x": 6, "y": 6}}}, {"action": {"id": "ACTION6", "data": {"x": 6, "y": 6}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 19, "y": 20}}}, {"action": {"id": "ACTION6", "data": {"x": 19, "y": 26}}}, {"action": {"id": "ACTION6", "data": {"x": 13, "y": 26}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 8, "y": 8}}}, {"action": {"id": "ACTION6", "data": {"x": 8, "y": 8}}}, {"action": {"id": "ACTION6", "data": {"x": 8, "y": 8}}}, {"action": {"id": "ACTION6", "data": {"x": 8, "y": 8}}}, {"action": {"id": "ACTION6", "data": {"x": 8, "y": 8}}}, {"action": {"id": "ACTION6", "data": {"x": 8, "y": 8}}}, {"action": {"id": "ACTION6", "data": {"x": 8, "y": 8}}}, {"action": {"id": "ACTION6", "data": {"x": 8, "y": 8}}}, {"action": {"id": "ACTION6", "data": {"x": 8, "y": 8}}}, {"action": {"id": "ACTION6", "data": {"x": 8, "y": 8}}}, {"action": {"id": "ACTION6", "data": {"x": 8, "y": 8}}}, {"action": {"id": "ACTION6", "data": {"x": 8, "y": 8}}}, {"action": {"id": "ACTION6", "data": {"x": 8, "y": 8}}}, {"action": {"id": "ACTION6", "data": {"x": 8, "y": 8}}}, {"action": {"id": "ACTION6", "data": {"x": 8, "y": 8}}}, {"action": {"id": "ACTION6", "data": {"x": 8, "y": 8}}}, {"action": {"id": "ACTION6", "data": {"x": 8, "y": 8}}}, {"action": {"id": "ACTION6", "data": {"x": 8, "y": 8}}}, {"action": {"id": "ACTION6", "data": {"x": 8, "y": 8}}}, {"action": {"id": "ACTION6", "data": {"x": 8, "y": 8}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 5, "y": 5}}}, {"action": {"id": "ACTION6", "data": {"x": 5, "y": 5}}}, {"action": {"id": "ACTION6", "data": {"x": 5, "y": 5}}}, {"action": {"id": "ACTION6", "data": {"x": 5, "y": 5}}}, {"action": {"id": "ACTION6", "data": {"x": 5, "y": 5}}}, {"action": {"id": "ACTION6", "data": {"x": 5, "y": 5}}}, {"action": {"id": "ACTION6", "data": {"x": 5, "y": 5}}}, {"action": {"id": "ACTION6", "data": {"x": 5, "y": 5}}}, {"action": {"id": "ACTION6", "data": {"x": 5, "y": 5}}}, {"action": {"id": "ACTION6", "data": {"x": 5, "y": 5}}}, {"action": {"id": "ACTION6", "data": {"x": 5, "y": 5}}}, {"action": {"id": "ACTION6", "data": {"x": 5, "y": 5}}}, {"action": {"id": "ACTION6", "data": {"x": 5, "y": 5}}}, {"action": {"id": "ACTION6", "data": {"x": 5, "y": 5}}}, {"action": {"id": "ACTION6", "data": {"x": 5, "y": 5}}}, {"action": {"id": "ACTION6", "data": {"x": 5, "y": 5}}}, {"action": {"id": "ACTION6", "data": {"x": 5, "y": 5}}}, {"action": {"id": "ACTION6", "data": {"x": 5, "y": 5}}}, {"action": {"id": "ACTION6", "data": {"x": 5, "y": 5}}}, {"action": {"id": "ACTION6", "data": {"x": 5, "y": 5}}}, {"action": {"id": "ACTION6", "data": {"x": 5, "y": 5}}}, {"action": {"id": "ACTION6", "data": {"x": 5, "y": 5}}}, {"action": {"id": "ACTION6", "data": {"x": 5, "y": 5}}}, {"action": {"id": "ACTION6", "data": {"x": 5, "y": 5}}}, {"action": {"id": "ACTION6", "data": {"x": 5, "y": 5}}}]}, {"game_id": "dc22-fdcac232", "history": [{"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 14, "y": 31}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 8, "y": 35}}}, {"action": {"id": "ACTION6", "data": {"x": 50, "y": 20}}}, {"action": {"id": "ACTION6", "data": {"x": 50, "y": 20}}}, {"action": {"id": "ACTION6", "data": {"x": 48, "y": 37}}}, {"action": {"id": "ACTION6", "data": {"x": 48, "y": 37}}}, {"action": {"id": "ACTION6", "data": {"x": 48, "y": 37}}}, {"action": {"id": "ACTION6", "data": {"x": 48, "y": 37}}}, {"action": {"id": "ACTION6", "data": {"x": 50, "y": 20}}}, {"action": {"id": "ACTION6", "data": {"x": 50, "y": 20}}}, {"action": {"id": "ACTION6", "data": {"x": 50, "y": 20}}}, {"action": {"id": "ACTION6", "data": {"x": 48, "y": 37}}}, {"action": {"id": "ACTION6", "data": {"x": 50, "y": 20}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 50, "y": 20}}}, {"action": {"id": "ACTION6", "data": {"x": 48, "y": 37}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 52, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 52, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 52, "y": 40}}}, {"action": {"id": "ACTION6", "data": {"x": 52, "y": 40}}}, {"action": {"id": "ACTION6", "data": {"x": 17, "y": 53}}}, {"action": {"id": "ACTION6", "data": {"x": 52, "y": 40}}}, {"action": {"id": "ACTION6", "data": {"x": 52, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 52, "y": 40}}}, {"action": {"id": "ACTION6", "data": {"x": 52, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 52, "y": 40}}}, {"action": {"id": "ACTION6", "data": {"x": 52, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 52, "y": 40}}}, {"action": {"id": "ACTION6", "data": {"x": 52, "y": 22}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 52, "y": 22}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 52, "y": 40}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 52, "y": 22}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 52, "y": 22}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 52, "y": 31}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 52, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 52, "y": 31}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 52, "y": 40}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 52, "y": 31}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 51, "y": 18}}}, {"action": {"id": "ACTION6", "data": {"x": 51, "y": 27}}}, {"action": {"id": "ACTION6", "data": {"x": 51, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 51, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 51, "y": 36}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 51, "y": 18}}}, {"action": {"id": "ACTION6", "data": {"x": 51, "y": 18}}}, {"action": {"id": "ACTION6", "data": {"x": 51, "y": 27}}}, {"action": {"id": "ACTION6", "data": {"x": 51, "y": 27}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 51, "y": 18}}}, {"action": {"id": "ACTION6", "data": {"x": 51, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 51, "y": 36}}}]}, {"game_id": "cd82-fb555c5d", "history": [{"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 38}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 28}}}, {"action": {"id": "ACTION6", "data": {"x": 37, "y": 4}}}, {"action": {"id": "ACTION6", "data": {"x": 43, "y": 4}}}, {"action": {"id": "ACTION6", "data": {"x": 43, "y": 7}}}, {"action": {"id": "ACTION6", "data": {"x": 7, "y": 5}}}, {"action": {"id": "ACTION6", "data": {"x": 7, "y": 10}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 24}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 50}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 38}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 40}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 33}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 43, "y": 4}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 37, "y": 4}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 43, "y": 4}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 41}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 41}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 28}}}, {"action": {"id": "ACTION6", "data": {"x": 26, "y": 28}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 37, "y": 4}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 43, "y": 4}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 37, "y": 4}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 43, "y": 4}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 37, "y": 4}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 43, "y": 4}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 37, "y": 4}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 43, "y": 4}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 37, "y": 4}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 46, "y": 4}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 4}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 38, "y": 1}}}]}, {"game_id": "ft09-0d8bbf25", "history": [{"action": {"id": "ACTION6", "data": {"x": 38, "y": 38}}}, {"action": {"id": "ACTION6", "data": {"x": 38, "y": 38}}}, {"action": {"id": "ACTION6", "data": {"x": 39, "y": 39}}}, {"action": {"id": "ACTION6", "data": {"x": 39, "y": 47}}}, {"action": {"id": "ACTION6", "data": {"x": 39, "y": 55}}}, {"action": {"id": "ACTION6", "data": {"x": 55, "y": 47}}}, {"action": {"id": "ACTION6", "data": {"x": 22, "y": 17}}}, {"action": {"id": "ACTION6", "data": {"x": 23, "y": 25}}}, {"action": {"id": "ACTION6", "data": {"x": 39, "y": 25}}}, {"action": {"id": "ACTION6", "data": {"x": 23, "y": 33}}}, {"action": {"id": "ACTION6", "data": {"x": 39, "y": 33}}}, {"action": {"id": "ACTION6", "data": {"x": 23, "y": 49}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 49}}}, {"action": {"id": "ACTION6", "data": {"x": 23, "y": 7}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 7}}}, {"action": {"id": "ACTION6", "data": {"x": 39, "y": 7}}}, {"action": {"id": "ACTION6", "data": {"x": 23, "y": 15}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 23}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 39}}}, {"action": {"id": "ACTION6", "data": {"x": 23, "y": 47}}}, {"action": {"id": "ACTION6", "data": {"x": 23, "y": 55}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 55}}}, {"action": {"id": "ACTION6", "data": {"x": 39, "y": 55}}}, {"action": {"id": "ACTION6", "data": {"x": 15, "y": 23}}}, {"action": {"id": "ACTION6", "data": {"x": 15, "y": 31}}}, {"action": {"id": "ACTION6", "data": {"x": 47, "y": 31}}}, {"action": {"id": "ACTION6", "data": {"x": 47, "y": 39}}}, {"action": {"id": "ACTION6", "data": {"x": 15, "y": 17}}}, {"action": {"id": "ACTION6", "data": {"x": 15, "y": 17}}}, {"action": {"id": "ACTION6", "data": {"x": 15, "y": 17}}}, {"action": {"id": "ACTION6", "data": {"x": 15, "y": 17}}}, {"action": {"id": "ACTION6", "data": {"x": 15, "y": 17}}}, {"action": {"id": "ACTION6", "data": {"x": 15, "y": 17}}}, {"action": {"id": "ACTION6", "data": {"x": 23, "y": 17}}}, {"action": {"id": "ACTION6", "data": {"x": 23, "y": 17}}}, {"action": {"id": "ACTION6", "data": {"x": 23, "y": 25}}}, {"action": {"id": "ACTION6", "data": {"x": 23, "y": 33}}}, {"action": {"id": "ACTION6", "data": {"x": 23, "y": 33}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 41}}}, {"action": {"id": "ACTION6", "data": {"x": 23, "y": 49}}}, {"action": {"id": "ACTION6", "data": {"x": 23, "y": 49}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 49}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 49}}}, {"action": {"id": "ACTION6", "data": {"x": 39, "y": 49}}}, {"action": {"id": "ACTION6", "data": {"x": 39, "y": 49}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 17}}}, {"action": {"id": "ACTION6", "data": {"x": 47, "y": 17}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 25}}}, {"action": {"id": "ACTION6", "data": {"x": 47, "y": 25}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 33}}}, {"action": {"id": "ACTION6", "data": {"x": 39, "y": 33}}}, {"action": {"id": "ACTION6", "data": {"x": 9, "y": 23}}}, {"action": {"id": "ACTION6", "data": {"x": 9, "y": 23}}}, {"action": {"id": "ACTION6", "data": {"x": 9, "y": 23}}}, {"action": {"id": "ACTION6", "data": {"x": 9, "y": 23}}}, {"action": {"id": "ACTION6", "data": {"x": 25, "y": 15}}}, {"action": {"id": "ACTION6", "data": {"x": 25, "y": 15}}}, {"action": {"id": "ACTION6", "data": {"x": 33, "y": 7}}}, {"action": {"id": "ACTION6", "data": {"x": 25, "y": 15}}}, {"action": {"id": "ACTION6", "data": {"x": 17, "y": 23}}}, {"action": {"id": "ACTION6", "data": {"x": 33, "y": 23}}}, {"action": {"id": "ACTION6", "data": {"x": 49, "y": 23}}}, {"action": {"id": "ACTION6", "data": {"x": 25, "y": 31}}}, {"action": {"id": "ACTION6", "data": {"x": 17, "y": 39}}}, {"action": {"id": "ACTION6", "data": {"x": 33, "y": 39}}}, {"action": {"id": "ACTION6", "data": {"x": 49, "y": 39}}}, {"action": {"id": "ACTION6", "data": {"x": 41, "y": 47}}}, {"action": {"id": "ACTION6", "data": {"x": 17, "y": 55}}}, {"action": {"id": "ACTION6", "data": {"x": 33, "y": 55}}}, {"action": {"id": "ACTION6", "data": {"x": 41, "y": 39}}}, {"action": {"id": "ACTION6", "data": {"x": 25, "y": 7}}}, {"action": {"id": "ACTION6", "data": {"x": 17, "y": 15}}}, {"action": {"id": "ACTION6", "data": {"x": 33, "y": 15}}}, {"action": {"id": "ACTION6", "data": {"x": 41, "y": 47}}}, {"action": {"id": "ACTION6", "data": {"x": 17, "y": 31}}}, {"action": {"id": "ACTION6", "data": {"x": 33, "y": 31}}}, {"action": {"id": "ACTION6", "data": {"x": 41, "y": 47}}}, {"action": {"id": "ACTION6", "data": {"x": 33, "y": 47}}}, {"action": {"id": "ACTION6", "data": {"x": 49, "y": 47}}}, {"action": {"id": "ACTION6", "data": {"x": 41, "y": 55}}}, {"action": {"id": "ACTION6", "data": {"x": 7, "y": 17}}}, {"action": {"id": "ACTION6", "data": {"x": 23, "y": 25}}}, {"action": {"id": "ACTION6", "data": {"x": 15, "y": 17}}}]}, {"game_id": "g50t-5849a774", "history": [{"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "RESET", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}]}, {"game_id": "ls20-9607627b", "history": [{"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "RESET", "data": {}}}]}, {"game_id": "re86-8af5384d", "history": [{"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}]}, {"game_id": "s5i5-18d95033", "history": [{"action": {"id": "ACTION6", "data": {"x": 30, "y": 10}}}, {"action": {"id": "ACTION6", "data": {"x": 39, "y": 21}}}, {"action": {"id": "ACTION6", "data": {"x": 5, "y": 5}}}, {"action": {"id": "ACTION6", "data": {"x": 52, "y": 9}}}, {"action": {"id": "ACTION6", "data": {"x": 10, "y": 51}}}, {"action": {"id": "ACTION6", "data": {"x": 42, "y": 18}}}, {"action": {"id": "ACTION6", "data": {"x": 24, "y": 35}}}, {"action": {"id": "ACTION6", "data": {"x": 10, "y": 27}}}, {"action": {"id": "ACTION6", "data": {"x": 27, "y": 10}}}, {"action": {"id": "ACTION6", "data": {"x": 39, "y": 21}}}, {"action": {"id": "ACTION6", "data": {"x": 23, "y": 38}}}, {"action": {"id": "ACTION6", "data": {"x": 23, "y": 38}}}, {"action": {"id": "ACTION6", "data": {"x": 23, "y": 38}}}, {"action": {"id": "ACTION6", "data": {"x": 45, "y": 21}}}, {"action": {"id": "ACTION6", "data": {"x": 45, "y": 21}}}, {"action": {"id": "ACTION6", "data": {"x": 45, "y": 21}}}, {"action": {"id": "ACTION6", "data": {"x": 45, "y": 21}}}, {"action": {"id": "ACTION6", "data": {"x": 23, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 23, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 23, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 45, "y": 21}}}, {"action": {"id": "ACTION6", "data": {"x": 45, "y": 21}}}, {"action": {"id": "ACTION6", "data": {"x": 45, "y": 21}}}, {"action": {"id": "ACTION6", "data": {"x": 23, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 23, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 23, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 23, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 45, "y": 21}}}, {"action": {"id": "ACTION6", "data": {"x": 23, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 12, "y": 57}}}, {"action": {"id": "ACTION6", "data": {"x": 21, "y": 57}}}, {"action": {"id": "ACTION6", "data": {"x": 27, "y": 57}}}, {"action": {"id": "ACTION6", "data": {"x": 57, "y": 57}}}, {"action": {"id": "ACTION6", "data": {"x": 12, "y": 57}}}, {"action": {"id": "ACTION6", "data": {"x": 12, "y": 57}}}, {"action": {"id": "ACTION6", "data": {"x": 12, "y": 57}}}, {"action": {"id": "ACTION6", "data": {"x": 12, "y": 57}}}, {"action": {"id": "ACTION6", "data": {"x": 12, "y": 57}}}, {"action": {"id": "ACTION6", "data": {"x": 12, "y": 57}}}, {"action": {"id": "ACTION6", "data": {"x": 12, "y": 57}}}, {"action": {"id": "ACTION6", "data": {"x": 12, "y": 57}}}, {"action": {"id": "ACTION6", "data": {"x": 21, "y": 57}}}, {"action": {"id": "ACTION6", "data": {"x": 51, "y": 57}}}, {"action": {"id": "ACTION6", "data": {"x": 27, "y": 57}}}, {"action": {"id": "ACTION6", "data": {"x": 27, "y": 57}}}, {"action": {"id": "ACTION6", "data": {"x": 27, "y": 57}}}, {"action": {"id": "ACTION6", "data": {"x": 27, "y": 57}}}, {"action": {"id": "ACTION6", "data": {"x": 27, "y": 57}}}, {"action": {"id": "ACTION6", "data": {"x": 27, "y": 57}}}, {"action": {"id": "ACTION6", "data": {"x": 27, "y": 57}}}, {"action": {"id": "ACTION6", "data": {"x": 27, "y": 57}}}, {"action": {"id": "ACTION6", "data": {"x": 42, "y": 57}}}, {"action": {"id": "ACTION6", "data": {"x": 42, "y": 57}}}, {"action": {"id": "ACTION6", "data": {"x": 42, "y": 57}}}, {"action": {"id": "ACTION6", "data": {"x": 42, "y": 57}}}, {"action": {"id": "ACTION6", "data": {"x": 57, "y": 57}}}, {"action": {"id": "ACTION6", "data": {"x": 57, "y": 57}}}, {"action": {"id": "ACTION6", "data": {"x": 57, "y": 57}}}, {"action": {"id": "ACTION6", "data": {"x": 57, "y": 57}}}, {"action": {"id": "ACTION6", "data": {"x": 57, "y": 57}}}, {"action": {"id": "ACTION6", "data": {"x": 57, "y": 57}}}, {"action": {"id": "ACTION6", "data": {"x": 48, "y": 48}}}, {"action": {"id": "ACTION6", "data": {"x": 16, "y": 48}}}, {"action": {"id": "ACTION6", "data": {"x": 36, "y": 57}}}, {"action": {"id": "ACTION6", "data": {"x": 36, "y": 57}}}, {"action": {"id": "ACTION6", "data": {"x": 36, "y": 57}}}, {"action": {"id": "ACTION6", "data": {"x": 36, "y": 57}}}, {"action": {"id": "ACTION6", "data": {"x": 36, "y": 57}}}, {"action": {"id": "ACTION6", "data": {"x": 36, "y": 57}}}, {"action": {"id": "ACTION6", "data": {"x": 36, "y": 57}}}, {"action": {"id": "ACTION6", "data": {"x": 36, "y": 57}}}, {"action": {"id": "ACTION6", "data": {"x": 36, "y": 57}}}, {"action": {"id": "ACTION6", "data": {"x": 36, "y": 57}}}, {"action": {"id": "ACTION6", "data": {"x": 36, "y": 57}}}, {"action": {"id": "ACTION6", "data": {"x": 36, "y": 57}}}, {"action": {"id": "ACTION6", "data": {"x": 36, "y": 57}}}, {"action": {"id": "ACTION6", "data": {"x": 36, "y": 57}}}, {"action": {"id": "ACTION6", "data": {"x": 48, "y": 48}}}, {"action": {"id": "ACTION6", "data": {"x": 48, "y": 48}}}, {"action": {"id": "ACTION6", "data": {"x": 48, "y": 48}}}, {"action": {"id": "ACTION6", "data": {"x": 48, "y": 48}}}, {"action": {"id": "ACTION6", "data": {"x": 48, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 36, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 42, "y": 48}}}, {"action": {"id": "ACTION6", "data": {"x": 34, "y": 48}}}, {"action": {"id": "ACTION6", "data": {"x": 34, "y": 48}}}, {"action": {"id": "ACTION6", "data": {"x": 34, "y": 48}}}, {"action": {"id": "ACTION6", "data": {"x": 34, "y": 48}}}, {"action": {"id": "ACTION6", "data": {"x": 34, "y": 48}}}, {"action": {"id": "ACTION6", "data": {"x": 34, "y": 48}}}, {"action": {"id": "ACTION6", "data": {"x": 34, "y": 48}}}]}, {"game_id": "sb26-7fbdac44", "history": [{"action": {"id": "ACTION6", "data": {"x": 35, "y": 58}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 35, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 22, "y": 29}}}, {"action": {"id": "ACTION6", "data": {"x": 19, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 29}}}, {"action": {"id": "ACTION6", "data": {"x": 43, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 34, "y": 29}}}, {"action": {"id": "ACTION6", "data": {"x": 27, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 29}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 22, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 17, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 10, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 22}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 45, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 22, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 24, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 52, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 34, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 38, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 36}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 22, "y": 22}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 22, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 17, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 10, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 22, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 45, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 24, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 34, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 52, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 38, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 24, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 22, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 52, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 38, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 22, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 17, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 10, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 34, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 45, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 36}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 22, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 24, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 52, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 38, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 22, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 17, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 34, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 10, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 45, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 10, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 22, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 45, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 24, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 24, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 22, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 52, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 38, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 34, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 36}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 22, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 10, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 45, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 24, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 52, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 34, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 38, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 22, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 17, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 10, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 45, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 22, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 24, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 52, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 34, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 38, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 36}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 22, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 17, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 10, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 22, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 45, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 24, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 34, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 52, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 38, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 10, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 22, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 45, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 24, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 22}}}, {"action": {"id": "RESET", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 18, "y": 18}}}, {"action": {"id": "ACTION6", "data": {"x": 22, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 34, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 38, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 22, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 45, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 10, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 17, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 22, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 24, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 34, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 52, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 36}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 22, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 38, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 45, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 10, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 22, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 17, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 34, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 24, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 52, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 10, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 22, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 17, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 24, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 22, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 38, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 45, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 34, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 52, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 36}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 22, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 10, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 17, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 24, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 22, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 38, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 34, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 45, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 52, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 38, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 22, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 52, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 24, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 45, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 22, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 10, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 17, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 34, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 36}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 22, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 38, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 52, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 24, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 22, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 45, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 10, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 34, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 17, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 22, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 17, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 10, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 45, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 22, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 24, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 52, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 34, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 38, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 22, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 17, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 10, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 22, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 45, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 24, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 34, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 52, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 38, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 22, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 17, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 10, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 45, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 22, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 24, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 36}}}, {"action": {"id": "RESET", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 10, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 22, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 17, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 45, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 22, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 24, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 52, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 34, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 38, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 22, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 10, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 24, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 22, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 17, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 52, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 38, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 24, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 10, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 34, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 52, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 17, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 34, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 38, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 38, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 22, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 52, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 24, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 22, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 45, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 22, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 10, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 17, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 34, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 17, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 10, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 34, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 45, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 22, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 38, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 10, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 22, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 52, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 17, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 24, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 38, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 22, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 52, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 34, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 10, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 24, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 34, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 22, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 10, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 34, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 22, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 22, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 17, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 10, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 52, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 17, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 22, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 17, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 10, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 22, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 38, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 45, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 22, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 52, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 24, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 34, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 24, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 52, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 34, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 17, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 38, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 36}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 10, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 34, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 52, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 38, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 22, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 17, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 10, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 22, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 45, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 10, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 34, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 22, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 17, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 10, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 34, "y": 36}}}, {"action": {"id": "RESET", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 22, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 17, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 10, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 45, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 22, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 24, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 52, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 34, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 38, "y": 58}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 34, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 34, "y": 32}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 36}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 22, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 22, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 22, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 34, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 22, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 34, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 22, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 34, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 34, "y": 36}}}, {"action": {"id": "ACTION5", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 22, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 22}}}, {"action": {"id": "ACTION6", "data": {"x": 34, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 22}}}]}, {"game_id": "su15-1944f8ab", "history": [{"action": {"id": "ACTION6", "data": {"x": 4, "y": 59}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 30}}}, {"action": {"id": "ACTION6", "data": {"x": 48, "y": 15}}}, {"action": {"id": "ACTION6", "data": {"x": 10, "y": 53}}}, {"action": {"id": "ACTION6", "data": {"x": 10, "y": 53}}}, {"action": {"id": "ACTION6", "data": {"x": 12, "y": 51}}}, {"action": {"id": "ACTION6", "data": {"x": 46, "y": 17}}}, {"action": {"id": "ACTION6", "data": {"x": 42, "y": 21}}}, {"action": {"id": "ACTION6", "data": {"x": 22, "y": 41}}}, {"action": {"id": "ACTION6", "data": {"x": 14, "y": 49}}}, {"action": {"id": "ACTION6", "data": {"x": 26, "y": 37}}}, {"action": {"id": "ACTION6", "data": {"x": 18, "y": 45}}}, {"action": {"id": "ACTION6", "data": {"x": 20, "y": 43}}}, {"action": {"id": "ACTION6", "data": {"x": 22, "y": 41}}}, {"action": {"id": "ACTION6", "data": {"x": 24, "y": 39}}}, {"action": {"id": "ACTION6", "data": {"x": 26, "y": 37}}}, {"action": {"id": "ACTION6", "data": {"x": 28, "y": 35}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 33}}}, {"action": {"id": "ACTION6", "data": {"x": 32, "y": 31}}}, {"action": {"id": "ACTION6", "data": {"x": 34, "y": 29}}}, {"action": {"id": "ACTION6", "data": {"x": 36, "y": 27}}}, {"action": {"id": "ACTION6", "data": {"x": 38, "y": 25}}}, {"action": {"id": "ACTION6", "data": {"x": 40, "y": 23}}}, {"action": {"id": "ACTION6", "data": {"x": 42, "y": 21}}}, {"action": {"id": "ACTION6", "data": {"x": 44, "y": 19}}}, {"action": {"id": "ACTION6", "data": {"x": 18, "y": 37}}}, {"action": {"id": "ACTION6", "data": {"x": 37, "y": 40}}}, {"action": {"id": "ACTION6", "data": {"x": 14, "y": 55}}}, {"action": {"id": "ACTION6", "data": {"x": 49, "y": 54}}}, {"action": {"id": "ACTION6", "data": {"x": 33, "y": 27}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 4}}}, {"action": {"id": "ACTION6", "data": {"x": 1, "y": 1}}}, {"action": {"id": "ACTION6", "data": {"x": 5, "y": 1}}}, {"action": {"id": "ACTION6", "data": {"x": 9, "y": 1}}}, {"action": {"id": "ACTION6", "data": {"x": 13, "y": 1}}}, {"action": {"id": "ACTION6", "data": {"x": 17, "y": 38}}}, {"action": {"id": "ACTION6", "data": {"x": 17, "y": 40}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 50}}}, {"action": {"id": "ACTION6", "data": {"x": 34, "y": 37}}}, {"action": {"id": "ACTION6", "data": {"x": 35, "y": 38}}}, {"action": {"id": "ACTION6", "data": {"x": 33, "y": 35}}}, {"action": {"id": "ACTION6", "data": {"x": 33, "y": 32}}}, {"action": {"id": "ACTION6", "data": {"x": 33, "y": 29}}}, {"action": {"id": "ACTION6", "data": {"x": 20, "y": 37}}}, {"action": {"id": "ACTION6", "data": {"x": 23, "y": 34}}}, {"action": {"id": "ACTION6", "data": {"x": 26, "y": 31}}}, {"action": {"id": "ACTION6", "data": {"x": 17, "y": 52}}}, {"action": {"id": "ACTION6", "data": {"x": 19, "y": 48}}}, {"action": {"id": "ACTION6", "data": {"x": 21, "y": 44}}}, {"action": {"id": "ACTION6", "data": {"x": 23, "y": 40}}}, {"action": {"id": "ACTION6", "data": {"x": 24, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 24, "y": 36}}}, {"action": {"id": "ACTION6", "data": {"x": 24, "y": 36}}}, {"action": {"id": "RESET", "data": {}}}, {"action": {"id": "ACTION6", "data": {"x": 16, "y": 41}}}, {"action": {"id": "ACTION6", "data": {"x": 18, "y": 37}}}, {"action": {"id": "ACTION6", "data": {"x": 37, "y": 40}}}, {"action": {"id": "ACTION6", "data": {"x": 14, "y": 55}}}, {"action": {"id": "ACTION6", "data": {"x": 49, "y": 54}}}, {"action": {"id": "ACTION6", "data": {"x": 41, "y": 37}}}, {"action": {"id": "ACTION6", "data": {"x": 16, "y": 57}}}, {"action": {"id": "ACTION6", "data": {"x": 17, "y": 52}}}, {"action": {"id": "ACTION6", "data": {"x": 52, "y": 10}}}, {"action": {"id": "ACTION6", "data": {"x": 52, "y": 10}}}, {"action": {"id": "ACTION6", "data": {"x": 52, "y": 10}}}, {"action": {"id": "ACTION6", "data": {"x": 17, "y": 49}}}, {"action": {"id": "ACTION6", "data": {"x": 17, "y": 46}}}, {"action": {"id": "ACTION6", "data": {"x": 17, "y": 43}}}, {"action": {"id": "ACTION6", "data": {"x": 48, "y": 50}}}, {"action": {"id": "ACTION6", "data": {"x": 46, "y": 46}}}, {"action": {"id": "ACTION6", "data": {"x": 43, "y": 41}}}, {"action": {"id": "ACTION6", "data": {"x": 37, "y": 39}}}, {"action": {"id": "ACTION6", "data": {"x": 34, "y": 39}}}, {"action": {"id": "ACTION6", "data": {"x": 31, "y": 39}}}, {"action": {"id": "ACTION6", "data": {"x": 33, "y": 42}}}, {"action": {"id": "ACTION6", "data": {"x": 30, "y": 42}}}, {"action": {"id": "ACTION6", "data": {"x": 27, "y": 42}}}, {"action": {"id": "ACTION6", "data": {"x": 27, "y": 40}}}, {"action": {"id": "ACTION6", "data": {"x": 25, "y": 40}}}, {"action": {"id": "ACTION6", "data": {"x": 29, "y": 41}}}, {"action": {"id": "ACTION6", "data": {"x": 32, "y": 41}}}, {"action": {"id": "ACTION6", "data": {"x": 34, "y": 38}}}, {"action": {"id": "ACTION6", "data": {"x": 34, "y": 35}}}, {"action": {"id": "ACTION6", "data": {"x": 34, "y": 32}}}, {"action": {"id": "ACTION6", "data": {"x": 35, "y": 29}}}, {"action": {"id": "ACTION6", "data": {"x": 12, "y": 23}}}, {"action": {"id": "ACTION6", "data": {"x": 15, "y": 20}}}]}, {"game_id": "tr87-cd924810", "history": [{"action": {"id": "ACTION3", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION2", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION4", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}, {"action": {"id": "ACTION1", "data": {}}}]}]}')
_env_dir = None
for _p in Path("/kaggle/input").rglob("environment_files"):
    if _p.is_dir():
        _env_dir = str(_p); break
print("WMPROBE: environment_files =", _env_dir, flush=True)
_base = dict(os.environ, WM_URL="http://127.0.0.1:1234/v1/chat/completions", WM_MODEL="Qwen/Qwen3.8-Flash-Next-NVFP4",
             WM_MAX_TOKENS="12000", WM_MASK_HUD="1", WM_MODE="OFFLINE", WM_ENV_DIR=_env_dir or "",
             WM_INFER_SRC=str(BUNDLE_DIR / "src" / "ARC3-Inference"))
for _arm, _think in (("A", "1"), ("B", "0")):
    _t0 = time.time()
    _env = dict(_base, WM_THINK=_think, WM_OUT="/kaggle/working/wm_%s.json" % _arm)
    print("WMPROBE: плечо %s (рассуждение %s) старт" % (_arm, "вкл" if _think == "1" else "выкл"), flush=True)
    _r = subprocess.run([sys.executable, str(_W / "scripts/wm_cegis_probe.py"), "--games", "all", "--par", "8", "--train", "6"],
                        env=_env, cwd=str(_W), capture_output=True, text=True)
    Path("/kaggle/working/wm_%s.log" % _arm).write_text(_r.stdout + "\n--- stderr ---\n" + _r.stderr[-20000:])
    print(_r.stdout[-6000:], flush=True)
    if _r.returncode:
        print("WMPROBE: плечо %s упало: %s" % (_arm, _r.stderr[-3000:]), flush=True)
    print("WMPROBE: плечо %s готово за %.0f мин" % (_arm, (time.time() - _t0) / 60), flush=True)
